In [ ]:
/*
========================================================
MULTI-TENANCY PATTERN - SIMPLE HINGLISH NOTES
========================================================

1. MULTI-TENANCY KYA HAI?
--------------------------------------------------------
Simple bhasha mein: Ek hi software system, but multiple 
customers (tenants) use karte hain. Har tenant ka data, 
permissions, limits alag-alag rehne chahiye, even though 
wo backend infrastructure share kar rahe hain.

ANALOGY: Apartment building
- Building shared hai (foundation, lift, security)
- Har apartment ka apna lock, apna space hai
- Software mein ye boundary banti hai: login, tenant 
  context, database queries, resource limits se

TEXT DIAGRAM - MULTI TENANT:

        [ Shared Application ]
        /        |          \
   Tenant A   Tenant B    Tenant C
   (Acme)     (Globex)    (Initech)
        \        |          /
        [ Shared Database ]

TEXT DIAGRAM - SINGLE TENANT (compare karne ke liye):

  Tenant A -> App1 -> DB1   (apna alag sab kuch)
  Tenant B -> App2 -> DB2
  Tenant C -> App3 -> DB3

MULTI vs SINGLE TENANT - KAB KYA USE KARE:

  Cost              -> Multi = sasta | Single = mehenga
  Operations        -> Multi = 1 deployment | Single = N deployments
  Resource use      -> Multi = high (share hota) | Single = low (idle rehta)
  Customization     -> Multi = limited | Single = full freedom
  Isolation         -> Multi = logical (code se) | Single = physical (alag machine)
  Compliance        -> Multi = tough | Single = easy (regulated industries ke liye)

Simple rule: Zyada tenants + kam usage = Multi-tenant better.
Bada customer + strict compliance/data residency = Single-tenant better.


2. ISOLATION MODELS (Resources kaise baatein)
--------------------------------------------------------
Main sawal: kaunse resources share honge, kaunse alag-alag 
honge har tenant ke liye?

--------------------------------------------------------
2.1 SILO MODEL (Full Isolation - sab kuch alag)
--------------------------------------------------------
Har tenant ko apna dedicated server + apna database milta hai.

TEXT DIAGRAM:

  [Load Balancer]
       |
  -----------------------------
  |            |              |
[App-A]      [App-B]        [App-C]
  |            |              |
[DB-A]       [DB-B]         [DB-C]
Tenant A     Tenant B       Tenant C

ACHA (PROS):
- Sabse strong isolation (security + performance dono)
- Data residency / compliance easily handle hota
- Tenant apne hisab se customize kar sakta
- Noisy neighbor problem nahi hoti (kyunki resource share nahi)

BURA (CONS):
- Sabse mehenga (har tenant = alag resources)
- Jaise-jaise tenant badhte, operations complex ho jate
- Resources idle padhe reh sakte (kam utilization)
- Manage karna mushkil scale pe

KAB USE KARE: Jab tenant ko strict compliance, data residency,
ya heavy customization chahiye ho, aur cost matter nahi karta.

--------------------------------------------------------
2.2 POOL MODEL (Shared Everything - sab kuch shared)
--------------------------------------------------------
Sab tenants ek hi app, ek hi database, ek hi tables share 
karte hain. Isolation sirf "logical" hai - code/login se 
enforce hota hai.

TEXT DIAGRAM:

           [Load Balancer]
          /      |        \
     [App1]   [App2]    [AppN]
          \      |        /
        [ Shared Database ]
        Tenant A, B, C sab yahin

ACHA (PROS):
- Sabse sasta (sab resources share)
- Simple operations (ek hi deployment manage karna)
- High resource utilization
- Scale karna easy (bas naye app servers add karo)

BURA (CONS):
- Sabse weak isolation (sab data ek jagah)
- Noisy neighbor risk (ek tenant heavy load daale to sab pe 
  asar)
- Access control ka dhyan har query mein rakhna padta
- Strict compliance ke liye mushkil

KAB USE KARE: Jab bahut saare chhote/medium tenants hain, 
cost important hai, aur logical isolation kaafi hai.

--------------------------------------------------------
2.3 BRIDGE MODEL (Hybrid - dono ka mix)
--------------------------------------------------------
Zyada tenants pooled infra mein rehte, bade/regulated 
tenants ko strong isolation milta.

TEXT DIAGRAM:

            [Load Balancer]
           /              \
   STANDARD TIER (Pool)   PREMIUM TIER (Silo)
   [App1] [App2]           [Dedicated App]
      \    /                    |
   [Shared DB]            [Dedicated DB]
   Std Tenant1,2          Enterprise Tenant

ACHA (PROS):
- Flexibility - customer ki need aur paisa dene ki 
  capacity ke hisab se isolation
- Premium tenants ka revenue standard tenants ka cost 
  subsidize karta hai
- Growth ka migration path milta hai (chhota tenant bada 
  ho to upgrade ho sakta)

BURA (CONS):
- Implement/operate karna sabse complex
- Multiple code paths maintain karne padte
- Routing logic sahi se pehchan e ki kaunsa tenant kis 
  tier mein hai

KAB USE KARE: Jab customer base mixed ho - kuch chhote, 
kuch bade - aur growth path chahiye.


3. DATA ISOLATION STRATEGIES
--------------------------------------------------------
Ye sabse risky part hai. Ek galti aur Tenant A ka data 
Tenant B ko dikh sakta hai.

--------------------------------------------------------
3.1 SEPARATE DATABASES (Har tenant ka apna DB)
--------------------------------------------------------
TEXT DIAGRAM:

Request "X-Tenant-ID: acme"
        |
  [Routing Logic] -- lookup connection -->
        |
  ---------------------------------------
  | tenant_acme_db | tenant_globex_db | tenant_initech_db |
  | users,orders   | users,orders     | users,orders      |
  ---------------------------------------

ACHA: Sabse strong isolation, backup/restore alag-alag 
easy, tenant delete karna simple.
BURA: Sabse mehenga (N databases), connection pooling 
complex, cross-tenant analytics ke liye extra kaam, 
migrations N baar chalani padti.

--------------------------------------------------------
3.2 SEPARATE SCHEMAS (Ek DB, alag schema per tenant)
--------------------------------------------------------
TEXT DIAGRAM:

  [ One Database Instance ]
   schema: tenant_acme    -> users, orders, products
   schema: tenant_globex  -> users, orders, products
   schema: tenant_initech -> users, orders, products

  Command: SET search_path TO tenant_acme

PostgreSQL mein "search_path" set karke tenant ka schema 
select hota hai. IMPORTANT: connection pooling use kar rahe 
ho to search_path reset karna mat bhoolna, warna ek tenant 
ka context dusre request mein leak ho sakta.

Schema ka naam hamesha trusted metadata se aana chahiye, 
URL ya user input se seedha nahi lena.

ACHA: Database schema level pe achi isolation, ek hi DB 
instance manage karna, cross-tenant analytics thoda easy.
BURA: Phir bhi N schemas manage karne, kuch databases 
schema count limit karte, migrations phir bhi per-tenant 
chalani padti.

--------------------------------------------------------
3.3 SHARED TABLES with tenant_id COLUMN
--------------------------------------------------------
Sab tenant ek hi table share karte, but ek column "tenant_id" 
batata hai ki row kiski hai.

TEXT DIAGRAM - users table:

  tenant_id | user_id | name
  ----------|---------|------
  acme      |   1     | Alice
  acme      |   2     | Bob
  globex    |   1     | Carol
  initech   |   1     | Dave

RULE: Har tenant-scoped query mein tenant_id hona ZAROORI 
hai, warna data leak ho sakta.

ACHA: Sabse sasta, ek hi schema manage karna, cross-tenant 
analytics easy, ek hi migration chalti.
BURA: Sabse weak isolation, har jagah tenant_id enforce 
karna padta, bug hui to data leak ho sakta, badi tables 
complex indexing maangti.

Ye wahi jagah hai jaha "normal bug" security incident ban 
jaata hai. Isliye application-level checks + database 
row-level security + cross-tenant tests - teeno layer 
lagao.

--------------------------------------------------------
3.4 COMPARISON - Isolation vs Cost Spectrum
--------------------------------------------------------
TEXT DIAGRAM:

  MORE ISOLATION <---------------------> MORE EFFICIENCY
  HIGHER COST    <---------------------> LOWER COST

  Separate DB  --  Separate Schema  --  Shared Tables+ID

TABLE:
  Strategy          Isolation | Cost   | Complexity | Best For
  Separate DB       Highest   | Highest| Medium     | Enterprise/regulated
  Separate Schema   High      | Medium | Medium     | Mid-market SaaS
  Shared Tables     Lowest    | Lowest | Low        | Self-service/high volume

COMMON INTERVIEW ANSWER: Self-service tenants ke liye pooled 
rakho, enterprise tenants ke liye separate DB/pool do, aur 
control plane flexible rakho taaki baad mein tenant move ho 
sake.


4. NOISY NEIGHBOR PROBLEM
--------------------------------------------------------
Jab tenants resources share karte hain, ek tenant ka heavy 
load dusre tenants ko slow kar sakta hai. Isko "noisy 
neighbor" bolte hain.

TEXT DIAGRAM:

  Tenant A (normal load) --\
  Tenant B (HEAVY QUERY) ----> [Shared Database - CPU 100%]
  Tenant C (normal load) --/
        |                              |
   Slow response                 Slow response
   (A aur C dono suffer karenge, 
    B ki wajah se)

COMMON SCENARIOS:
- Ek tenant heavy analytics query chalata, DB lock ho jata
- Ek tenant bulk import karta, sara disk I/O le leta
- Ek tenant viral ho jata, traffic flood kar deta
- Sab tenants ke scheduled jobs ek hi time pe chal jaate

SOLUTIONS (Multiple layers use karo):

--------------------------------------------------------
4.1 RESOURCE QUOTAS AND RATE LIMITING
--------------------------------------------------------
Har tenant ki tier ke hisab se limit set karo.

EXAMPLE (Acme Corp - Standard Tier):
  API requests     : 1000/min
  DB queries       : 500/min
  Storage          : 10 GB
  Concurrent users : 100
  Export rows      : 10,000/request

TEXT FLOW:
  Request -> Extract tenant_id -> Rate limit exceeded?
     -> YES -> 429 Too Many Requests
     -> NO  -> Process request

Rate limit counters usually fast shared store (jaise Redis) 
mein rakhte hain with expiry.

--------------------------------------------------------
4.2 QUERY GOVERNANCE
--------------------------------------------------------
Rate limit sirf "kitni requests" control karta, but ek 
heavy query bhi DB overload kar sakti. Isliye query ka 
shape/cost bhi control karo.

RULES:
  Max execution time     : 30 sec       -> long locks na ho
  Max rows scanned       : 1,000,000    -> full table scan na ho
  Max result set size    : 10,000 rows  -> memory na phate
  tenant_id WHERE clause : Required     -> cross-tenant access na ho
  SELECT * without LIMIT : Blocked      -> unbounded result na aaye

Ye especially report builders, exports, custom dashboards 
ke liye important hai.

--------------------------------------------------------
4.3 TENANT-AWARE RESOURCE POOLS
--------------------------------------------------------
Tier ke hisab se resource pools alag karo.

TEXT DIAGRAM:

              [Load Balancer]
             /              \
     STANDARD POOL         PREMIUM POOL
     Server1, Server2      Server1, Server2, Server3
          |                        |
     [Standard DB]           [Premium DB]
     (Standard Tenants)      (Enterprise Tenants)

--------------------------------------------------------
4.4 FAIR SCHEDULING
--------------------------------------------------------
Rate limiting ke baad bhi kuch tenants legit heavy traffic 
se system overwhelm kar sakte. Weighted fair queuing use 
karo.

TEXT DIAGRAM:

  Tenant A (weight 1): req1, req2
  Tenant B (weight 1): req1, req2, req3
  Tenant C (weight 2): req1
        |
  [Request Queue - Weighted Fair]
        |
  Order: A1, B1, C1, C2, A2, B2 ...
        |
  [Queue Processor] -> Process

Zyada weight (premium customers) ko proportionally zyada 
processing time milta, but free-tier bhi progress karta 
rehta, busy time mein bhi.


5. TENANT IDENTITY AND ROUTING
--------------------------------------------------------
Data isolate karne se pehle, har request ko pata hona 
chahiye "ye kis tenant ka request hai" - trusted tarike se.

--------------------------------------------------------
5.1 IDENTIFICATION METHODS
--------------------------------------------------------

(a) SUBDOMAIN-BASED
  URL: https://acme.slack.com/messages
  Flow: Parse subdomain "acme" -> Lookup tenant -> Set context
  ACHA: Clean URL, branding acha
  BURA: Wildcard DNS + SSL chahiye, subdomain conflicts 
        handle karne padte

(b) PATH-BASED
  URL: api.example.com/tenants/acme/orders
  Flow: Parse path "acme" -> Lookup tenant -> Set context
  ACHA: Simple, ek hi domain, no DNS complexity
  BURA: User ke liye less intuitive, URL lambi

(c) HEADER-BASED
  Request:
    GET /api/orders
    X-Tenant-ID: acme
    Authorization: Bearer ...
  Flow: Validate tenant matches token -> Set context
  ACHA: Clean URLs, programmatic access ke liye flexible
  BURA: Custom header handle karna padta, client bhool 
        sakte

(d) TOKEN-BASED (JWT Claims)
  JWT Payload:
    sub: user_123
    tenant_id: acme
    roles: admin
  Flow: Validate JWT -> Extract tenant_id -> Set context
  ACHA: Self-contained, tamper-evident (agar sign hai)
  BURA: Token size badhta, user tenant switch kare to 
        refresh chahiye, revocation handle karna padta

--------------------------------------------------------
5.2 TENANT CONTEXT PATTERN
--------------------------------------------------------
Tenant identify hone ke baad, ye context poori request 
journey mein carry hona chahiye - auth, business logic, 
DB, cache, logs, rate limits, background jobs - sabme.

TEXT FLOW:

  Request -> Middleware (Set Context)
          -> Controller (Pass Context)
          -> Service (Pass Context)
          -> Data Layer (Use Context)

  Context = { tenant_id: acme, tier: premium, limits: ... }

TAREEKE (3 patterns):

  1. Thread-Local Storage
     - Tenant context ek thread-local variable mein store
     - ACHA: Simple | BURA: Async operations ke saath break 
       ho sakta

  2. Explicit Parameter Passing
     - tenant_id ko har function mein parameter ki tarah 
       pass karo
     - ACHA: Explicit, async ke saath achhe se kaam karta
     - BURA: Verbose (bar bar likhna padta)

  3. Request-Scoped Dependency Injection
     - Tenant context ek request-scoped service ki tarah 
       inject hota
     - ACHA: Clean, testable | BURA: Framework pe depend 
       karta

--------------------------------------------------------
5.3 DATABASE QUERY ENFORCEMENT
--------------------------------------------------------
Sabse dangerous bug: koi access path jo tenant scope bhool 
jaye. Isko likhna hard banao, test karna easy banao.

APPROACH 1: Query Wrapper
  - Saara DB access ek function ke through jaye jo 
    tenant_id maange (bina tenant_id ke query hi nahi ban 
    sakti)

APPROACH 2: Row-Level Security (PostgreSQL)
  - Database khud enforce kare tenant isolation
  - Agar application code bhi tenant_id filter bhool jaye,
    tab bhi DB galat tenant ka data return nahi karega
  - Pooled connections use kar rahe ho to "SET LOCAL" 
    transaction ke andar use karo, ya session variable 
    reliably reset karo

APPROACH 3: ORM Scopes
  - Ek base scope banao jisse saari queries guzarni 
    zaroori ho (jaise "always filter by current tenant")

RULE: In teeno layers ko combine karo - tenant-aware data 
access layer + database RLS + cross-tenant tests.


6. SECURITY CONSIDERATIONS
--------------------------------------------------------
Security ka matlab hai: tenant boundary ko explicit aur 
testable banana. Har layer ko pata hona chahiye ki kis 
tenant ki request hai, aur boundary cross karne wali 
request reject karni chahiye.

--------------------------------------------------------
6.1 DATA LEAKAGE PREVENTION
--------------------------------------------------------
Sabse risky bug: Tenant A ka data Tenant B ko dikh jana. 
Isko sirf "query convention" mat samjho, isko authorization 
boundary treat karo.

LAYERS:
  Tenant-aware data access -> Base repo/ORM scope tenant_id 
                              maange
  Row-level security       -> Database khud tenant filter 
                              karega
  Code review checklist    -> Queries, caches, jobs review 
                              ho tenant scoping ke liye
  Automated testing         -> Cross-tenant access denied 
                              hona chahiye, test se verify
  Security testing          -> Regular audits mein tenant 
                              isolation shamil ho

--------------------------------------------------------
6.2 AUTHENTICATION AND AUTHORIZATION
--------------------------------------------------------
Authentication = "Tum kaun ho?"
Authorization = "Tum is tenant ke andar ye action kar sakte 
ho kya?"

TEXT FLOW:

  Request -> Authentication
     -> Invalid -> 401 Unauthorized
     -> Valid -> Tenant Match?
           -> No -> 403 Forbidden
           -> Yes -> Authorization
                 -> Denied -> 403 Forbidden
                 -> Allowed -> Process Request

3 STEPS ZAROORI:
  1. Authenticate the user - "Tum kaun ho?"
  2. Verify tenant membership - "Tum is tenant ke member 
     ho?"
  3. Authorize the action - "Tumhe ye action karne ki 
     permission hai?"

COMMON MISTAKE: URL ya header se tenant le lena bina 
authenticated identity se match kiye.

--------------------------------------------------------
6.3 ENCRYPTION
--------------------------------------------------------
  At rest    -> Envelope encryption; strict tiers ke liye 
                tenant-specific keys
  In transit -> TLS everywhere
  In use     -> Exposure minimize karo, sensitive data 
                kabhi log mat karo

TEXT DIAGRAM - Tenant Key Decryption Flow:

  [Master Key in HSM]
        |
   decrypts
        |
  [Encrypted Tenant Keys] (stored encrypted)
        |
   decrypts
        |
  [Tenant Data]

Ye pattern key exposure ka blast radius limit karta hai aur 
per-tenant key rotation support karta, but key management 
complexity badhati hai.

--------------------------------------------------------
6.4 AUDIT LOGGING
--------------------------------------------------------
Audit logs mein tenant context zaroor ho, taaki incidents, 
support cases, compliance review sahi se scope ho sake.

RULE: Audit logs ko bhi product data jaisi isolation milni 
chahiye. Tenant A, Tenant B ke audit logs nahi dekh sakta.


7. SCALING MULTI-TENANT SYSTEMS
--------------------------------------------------------
Scaling ka matlab: app tier stateless rakho, tenants ko 
smartly place karo, aur jab tenant ki size/need badle to 
unhe move kar sako.

--------------------------------------------------------
7.1 HORIZONTAL SCALING
--------------------------------------------------------
TEXT DIAGRAM:

        [Load Balancer]
       /      |        \
   [App1]  [App2]    [AppN]
       \      |        /
        [Cache]
           |
        [Database]

Sab app servers interchangeable hone chahiye. Tenant-specific 
state ek server mein mat rakho - request context, shared 
services, ya tenant metadata store mein rakho.

--------------------------------------------------------
7.2 DATABASE SCALING
--------------------------------------------------------
  Read Replicas       -> Write primary ko, read replicas se
                          (Best for: read-heavy workload)
  Tenant-based Sharding -> tenant_id = shard key
                          (Best for: high tenant count, no 
                          cross-tenant queries)
  Tiered Storage       -> Hot data SSD pe, warm normal 
                          storage pe, cold archive mein
                          (Best for: cost optimization)

Tenant-based sharding usually achha kaam karta hai kyunki 
zyada tar queries ek hi tenant tak scoped hoti hain.

--------------------------------------------------------
7.3 TENANT PLACEMENT
--------------------------------------------------------
  Round-robin      -> Even distribution, simple
  Size-based       -> Bade tenants ko dedicated shard
  Geography-based  -> Tenant ke data ke paas place karo 
                      (low latency)
  Activity-based   -> Active tenants spread karo, hotspot 
                      na bane

REBALANCING KAB ZAROORI:
- Ek shard hotspot ban jaye
- Chhota tenant bada ho jaye
- Naye shards cluster mein add ho

TEXT FLOW - Rebalancing:

  1. Create new home on target shard
  2. Sync data (CDC ya dual-write)
  3. Update routing table
  4. Cutover traffic
  5. Clean up old data

Goal: Online ya low-downtime movement - copy data, verify, 
switch routing, monitor, phir purana data clean karo.


8. INTERVIEW ANSWER FRAMEWORK
--------------------------------------------------------
Interview mein multi-tenancy aaye to seedha isolation 
models list mat karo. Risk boundaries se start karo.

STEPS:
  1. Tenant boundary identify karo - system ko kaise pata 
     chalta har request/job/cache/log/query kis tenant ka hai
  2. Sharing model choose karo - chhote tenants pool karo, 
     bade/regulated ko silo/strong isolation, dono ho to 
     bridge
  3. Data pehle protect karo - tenant-scoped queries, RLS, 
     tests, reviews explain karo
  4. Noisy neighbors control karo - rate limits, quotas, 
     query limits, fair queues, resource pools
  5. Movement plan karo - tenant metadata/routing control 
     plane mein rakho taaki shard/tier badal sake
  6. Operations tenant-aware banao - monitoring, audit 
     logs, billing, support tools, migrations sabme 
     tenant_id ho

ACHA ANSWER KYA COVER KARTA HAI:
- Tenant identity ko trusted context maanta hai, raw 
  request parameter nahi
- Shared tables vs separate schema vs separate database 
  vs dedicated infra ka tradeoff explain karta
- Data access, cache keys, queues, jobs, logs, support 
  tools, migrations - sab cover karta
- Noisy neighbors aur tenant movement plan karta, ye nahi 
  sochta ki sab tenant hamesha same size rahenge

COMMON MISTAKES (Candidates ki):
- "Authentication kaafi hai" bolna, tenant authorization 
  aur tenant-scoped data access bhool jaana
- tenant_id sirf HTTP request mein daalna, caches/jobs/
  support tools mein nahi
- Har customer ke liye "one database per tenant" bol dena 
  bina operational cost consider kiye
- Shared tables use karna bina RLS/query wrapper/tests/
  reviews ke

SAMPLE STRONG ANSWER:
"Main tenant context authentication + membership check se 
nikaalunga, phir usko database access, cache keys, queues, 
logs, rate limits sabse bind karunga. Zyada tenants ke liye 
pooled tier with shared tables + strong scoping controls 
(query wrappers, RLS, tests) use karunga. Enterprise/
regulated tenants ke liye dedicated database/pool dunga. 
Tenant placement control plane mein rakhunga taaki bade 
tenant baad mein move ho sake bina product code change 
kiye."


9. SUMMARY - KEY TAKEAWAYS
--------------------------------------------------------
- Multi-tenancy pehle "tenant-boundary problem" hai, baad 
  mein "infra-sharing problem"
- Isolation ek spectrum hai: Silo = strong isolation, 
  Pool = better efficiency, Bridge = middle ground
- Data isolation sabse risky area hai - shared tables tabhi 
  theek hain jab tenant scoping consistently enforce aur 
  test ho
- Tenant context HAR JAGAH flow hona chahiye - requests, 
  DB queries, caches, queues, logs, background jobs, 
  support tools
- Noisy neighbors ko control karne ke liye rate limits, 
  quotas, query governance, resource pools chahiye
- Tenant movement ke liye design karo - growing tenants 
  ko pooled se strong isolation mein move karna easy hona 
  chahiye, bina full rewrite ke

========================================================
END OF NOTES
========================================================
*/

In [ ]:
/**
 * 1. MULTI-TENANCY KYA HAI?
 *    - Multi-tenancy mein ek hi software system ko multiple customers use karte hain. 
 *    - Har customer ko "Tenant" kaha jata hai.
 *    - Example:
 *      - Tenant A = Acme company
 *      - Tenant B = Globex company
 *      - Tenant C = Initech company
 *    - Sab tenants same backend infrastructure share karte hain, lekin har tenant ka:
 *      - Data alag hona chahiye
 *      - Permissions alag honi chahiye
 *      - Resource limits alag ho sakti hain
 *    - Matlab Tenant A ka data Tenant B ko nahi dikhna chahiye.
 * 
 * 
 *    ANALOGY: Apartment Building
 *    - Multi-tenancy ko apartment building ki tarah samjho.
 *    - Apartment Building
 *              |
 *              |-- Building/Foundation shared hai
 *              |-- Lift shared hai
 *              |-- Security shared hai
 *              |
 *              |-- Apartment A -> apna lock + apna space
 *              |-- Apartment B -> apna lock + apna space
 *              |-- Apartment C -> apna lock + apna space
 * 
 *    - Software mein bhi same idea hota hai:
 *    - Shared Software System
 *              |
 *              |-- Tenant A -> apna data + permissions
 *              |-- Tenant B -> apna data + permissions
 *              |-- Tenant C -> apna data + permissions
 * 
 *    - Tenant ke beech ye boundary maintain karne ke liye use hota hai:
 *      - Login
 *      - Tenant Context
 *      - Database Queries
 *      - Resource Limits
*/

In [ ]:
/**
 * SINGLE-TENANT ARCHITECTURE
 * - Tenant A -> App 1 -> DB 1
 * - Tenant B -> App 2 -> DB 2
 * - Tenant C -> App 3 -> DB 3
 * - Yaha har tenant ka apna separate:
 *   - Application
 *   - Database
 *   - Infrastructure,
 *     ho sakta hai.
 * 
 * 
 * MULTI-TENANT ARCHITECTURE
 * 
 *             [ Shared Application ]
 *                       |
 *           -------------------------
 *           |           |           |
 *       Tenant A    Tenant B    Tenant C
 *        (Acme)     (Globex)    (Initech)
 *           |           |           |
 *           -------------------------
 *                       |
 *               [ Shared Database ]
 * 
 *    Yaha:
 *    - Application shared hai
 *    - Database bhi shared hai
 *    - Multiple tenants same infrastructure use kar rahe hain
 *    - Lekin logically har tenant ka data separate rakha jata hai
 * 
 * 
 * MULTI-TENANT vs SINGLE-TENANT:
 * 1. Cost
 *    - Multi-Tenant
 *      - Infrastructure share hota hai
 *      - Isliye generally sasta hota hai
 *    - Single-Tenant
 *      - Har tenant ke liye separate setup
 *      - Isliye generally mehenga hota hai
 * 
 * 2. Operations
 *    - Multi-Tenant : Ek shared deployment manage karna hota hai
 *    - Single-Tenant: N tenants hain to multiple deployments manage karne pad sakte hain
 * 
 * 3. Resource Usage
 *    - Multi-Tenant
 *      - Resources tenants ke beech share hote hain
 *      - Isliye resource utilization high ho sakta hai
 *    - Single-Tenant
 *      - Dedicated resources idle bhi reh sakte hain
 *      - Isliye utilization comparatively low ho sakta hai
 * 
 * 4. Customization
 *    - Multi-Tenant: Shared system hone ki wajah se customization limited hoti hai
 *    - Single-Tenant
 *      - Tenant ka separate system hota hai
 *      - Isliye customization ki freedom zyada hoti hai
 * 
 * 5. Isolation
 *    - Multi-Tenant
 *      - Mostly logical isolation hota hai
 *      - Code, tenant context aur database queries se separation hoti hai
 *    - Single-Tenant
 *      - Physical isolation ho sakta hai
 *      - Tenant ka infrastructure alag hota hai
 * 
 * 6. Compliance
 *    - Multi-Tenant: Compliance handle karna comparatively tough ho sakta hai
 *    - Single-Tenant
 *      - Dedicated infrastructure ki wajah se compliance easier ho sakti hai
 *      - Regulated industries ke liye useful ho sakta hai
 * 
 * 
 * SIMPLE RULE
 * a. Zyada Tenants + Har tenant ka kam usage
 *                       |
 *                       v
 *             Multi-Tenant Better
 * 
 * b. Bada Customer + Strict Compliance / Data Residency
 *                          |
 *                          v
 *                Single-Tenant Better
*/

In [ ]:
/**
 * 2.1 SILO MODEL (Full Isolation - sab kuch alag)
 * 
 * - Silo Model mein har tenant ko apna completely separate infrastructure diya jata hai.
 * - Har tenant ke paas hota hai:
 *   - Apna dedicated server
 *   - Apna application
 *   - Apna database
 * - Matlab Tenant A ka infrastructure Tenant B ya Tenant C ke saath share nahi hota.
 * 
 *                 [Load Balancer]
 *                       |
 *           -----------------------------
 *           |            |              |
 *        [App-A]      [App-B]        [App-C]
 *           |            |              |
 *        [DB-A]       [DB-B]         [DB-C]
 *           |            |              |
 *        Tenant A     Tenant B       Tenant C
 * 
 * Example:
 * - Tenant A -> App-A -> DB-A
 * - Tenant B -> App-B -> DB-B
 * - Tenant C -> App-C -> DB-C
 * - Har tenant ka complete setup alag hai.
 * 
 * 
 * PROS:
 * 1. Strong Isolation
 *    -> Har tenant ka infrastructure alag hota hai
 *    -> Security aur performance isolation strong hoti hai
 * 
 * 2. Data Residency / Compliance
 *    -> Tenant ka data dedicated environment mein hota hai
 *    -> Data residency aur compliance requirements handle karna easier hota hai
 * 
 * 3. Customization
 *    -> Har tenant apne requirements ke according system customize
 *       kar sakta hai
 * 
 * 4. Noisy Neighbor Problem Nahi
 *    -> Tenants resources share nahi karte
 *    -> Ek tenant ke heavy usage ka doosre tenant par direct impact nahi hota
 * 
 * 
 * CONS:
 * 1. Sabse Mehenga
 *    -> Har tenant ke liye separate resources chahiye
 *    -> Tenants badhne ke saath infrastructure cost bhi badhti hai
 * 
 * 2. Operations Complex
 *    -> Har tenant ka separate setup manage karna padta hai
 *    -> Jaise-jaise tenants badhte hain, management difficult hota jata hai
 * 
 * 3. Resources Idle Ho Sakte Hain
 *    -> Agar koi tenant kam resources use karta hai,
 *       uske dedicated resources idle reh sakte hain
 *    -> Isliye resource utilization low ho sakta hai
 * 
 * 4. Large Scale Par Manage Karna Mushkil
 *    -> Bahut saare tenants hone par bahut saare
 *       applications, servers aur databases manage karne padte hain
 * 
 * 
 * KAB USE KARE?
 * - Silo Model tab use kare jab tenant ko:
 *   - Strict compliance chahiye
 *   - Data residency requirements hain
 *   - Heavy customization chahiye
 * - Aur infrastructure ki high cost acceptable ho.
 * 
 * 
 * Simple Rule:
 * > Strict Compliance
 *        +
 * > Data Residency
 *        +
 * > Heavy Customization
 *        +
 * > Cost is not the main concern
 *        |
 *        v
 *    SILO MODEL
*/

In [ ]:
/*
--------------------------------------------------------
2.2 POOL MODEL (Shared Everything - sab kuch shared)
--------------------------------------------------------

Pool Model mein sab tenants same infrastructure share karte hain.

Sab tenants use karte hain:

- Same application
- Same database
- Same database tables

Isolation physical nahi hota. Isolation sirf "logical" hota hai.

Matlab code aur login ke through decide hota hai ki
kaunsa tenant ka data access kar sakta hai.


TEXT DIAGRAM:

                  [Load Balancer]
                 /      |       \
             [App1]   [App2]    [AppN]
                 \      |       /
                  \     |      /
                [ Shared Database ]
                 /      |       \
            Tenant A  Tenant B  Tenant C

Sab tenants same shared database use kar rahe hain.


ACHA (PROS):
--------------------------------------------------------

1. Sabse Sasta
   -> Sab resources share hote hain
   -> Har tenant ke liye separate infrastructure nahi chahiye

2. Simple Operations
   -> Ek hi application deployment manage karna hota hai
   -> Har tenant ke liye separate deployment nahi chahiye

3. High Resource Utilization
   -> Resources sab tenants ke beech share hote hain
   -> Jo resources available hain unka better utilization ho sakta hai

4. Scale Karna Easy
   -> Load badhne par naye application servers add kar sakte hain
   -> Har tenant ke liye separate server setup nahi karna padta


BURA (CONS):
--------------------------------------------------------

1. Weak Isolation
   -> Sab tenants same database use karte hain
   -> Isliye isolation Silo Model se weak hota hai

2. Noisy Neighbor Risk
   -> Agar ek tenant bahut heavy load daalta hai,
      to doosre tenants bhi affect ho sakte hain

3. Har Query Mein Access Control
   -> Har database query mein ensure karna padta hai
      ki tenant sirf apna data access kare
   -> Galat query se doosre tenant ka data expose ho sakta hai

4. Strict Compliance Ke Liye Mushkil
   -> Sab tenants shared infrastructure use karte hain
   -> Isliye strict compliance requirements handle karna
      comparatively difficult ho sakta hai


KAB USE KARE?
--------------------------------------------------------

Pool Model tab use kare jab:

- Bahut saare small/medium tenants hain
- Cost important hai
- Logical isolation sufficient hai

Simple Rule:

Many Small/Medium Tenants
        +
Cost Important
        +
Logical Isolation Enough
        |
        v
    POOL MODEL
*/

In [ ]:
/*
--------------------------------------------------------
2.3 BRIDGE MODEL (Hybrid - dono ka mix)
--------------------------------------------------------

Bridge Model mein Multi-Tenant ke Pool Model aur
Silo Model dono ko combine kiya jata hai.

Zyada tenants shared/pooled infrastructure use karte hain,
lekin bade ya regulated tenants ko dedicated infrastructure
diya jata hai.

Matlab:

- Standard tenants -> Pool mein
- Premium/Enterprise tenants -> Dedicated/Silo setup mein


TEXT DIAGRAM:

                    [Load Balancer]
                   /              \
                  /                \
     STANDARD TIER (Pool)      PREMIUM TIER (Silo)
          |                         |
     [App1] [App2]             [Dedicated App]
          |                         |
     [Shared DB]              [Dedicated DB]
          |                         |
   Standard Tenant 1,2       Enterprise Tenant


STANDARD TIER:
-> Multiple standard tenants same infrastructure share karte hain
-> Shared application aur database use hota hai


PREMIUM TIER:
-> Premium/Enterprise tenant ko dedicated infrastructure milta hai
-> Dedicated application aur database ho sakta hai


ACHA (PROS):
--------------------------------------------------------

1. Flexibility
   -> Har customer ki need aur budget ke according
      isolation level choose kiya ja sakta hai

   -> Small customer -> Pool
   -> Large/regulated customer -> Dedicated/Silo


2. Premium Tenants Revenue Generate Karte Hain
   -> Premium tenants dedicated setup ke liye zyada pay kar sakte hain
   -> Ye extra revenue standard tenants ke infrastructure
      cost ko subsidize karne mein help kar sakta hai


3. Growth / Migration Path
   -> Customer initially standard pool mein start kar sakta hai
   -> Jaise-jaise customer bada hota hai,
      usko premium/dedicated tier mein migrate kiya ja sakta hai


BURA (CONS):
--------------------------------------------------------

1. Sabse Complex Implementation
   -> Pool aur Silo dono models ko support karna padta hai
   -> Isliye implementation aur operations complex ho jate hain


2. Multiple Code Paths
   -> Different tenant types ke liye different handling ho sakti hai
   -> Multiple code paths maintain karne padte hain


3. Routing Logic
   -> System ko correctly identify karna padta hai ki
      kaunsa tenant kis tier mein hai

   Example:

   Tenant A -> Standard Tier -> Shared App/DB
   Tenant B -> Premium Tier -> Dedicated App/DB


KAB USE KARE?
--------------------------------------------------------

Bridge Model tab use kare jab customer base mixed ho:

- Kuch small tenants hain
- Kuch large tenants hain
- Kuch tenants ko strict isolation chahiye
- Future mein tenants ko upgrade karne ka growth path chahiye


Simple Rule:

Small Tenants
      +
Large / Regulated Tenants
      +
Different Isolation Requirements
      +
Future Upgrade Path
      |
      v
  BRIDGE MODEL
*/

In [ ]:
/*
--------------------------------------------------------
3.1 SEPARATE DATABASES (Har Tenant Ka Apna DB)
--------------------------------------------------------

Is model mein har tenant ka apna separate database hota hai.

Matlab:

- Tenant A -> apna DB
- Tenant B -> apna DB
- Tenant C -> apna DB

Application shared ho sakti hai, lekin data ke liye
har tenant ka database alag hota hai.


TEXT DIAGRAM:

Request:
"X-Tenant-ID: acme"
        |
        v
  [Routing Logic]
        |
        |-- Tenant ID check karo
        |-- Tenant ke database ka connection find karo
        |
        v
  -------------------------------
  | tenant_acme_db               |
  | users, orders                |
  -------------------------------

  | tenant_globex_db             |
  | users, orders                |
  -------------------------------

  | tenant_initech_db            |
  | users, orders                |
  -------------------------------


FLOW:

Request
   |
   v
X-Tenant-ID: acme
   |
   v
[Routing Logic]
   |
   v
Tenant "acme" ka DB identify karo
   |
   v
[tenant_acme_db]
   |
   v
Users / Orders ka data


ACHA (PROS):
--------------------------------------------------------

1. Strong Isolation
   -> Har tenant ka database completely separate hota hai
   -> Isliye data isolation strong hoti hai

2. Backup / Restore Easy
   -> Har tenant ka database separately backup kiya ja sakta hai
   -> Kisi ek tenant ka database separately restore bhi kar sakte hain

3. Tenant Delete Karna Simple
   -> Kisi tenant ko remove karna ho to
      uska separate database delete kiya ja sakta hai


BURA (CONS):
--------------------------------------------------------

1. Sabse Mehenga
   -> Har tenant ke liye separate database chahiye
   -> N tenants = N databases
   -> Tenants badhne ke saath cost bhi badhti hai

2. Connection Pooling Complex
   -> Har tenant ke database ke liye connections manage karne padte hain
   -> Bahut saare tenants hone par connection management difficult ho sakta hai

3. Cross-Tenant Analytics Difficult
   -> Data different databases mein pada hai
   -> Sab tenants ka combined analytics karne ke liye
      extra work karna padta hai

4. Migrations N Baar Chalani Padti Hain
   -> Agar database schema change karna hai,
      to migration har tenant ke database par run karni pad sakti hai

   Example:

   Tenant A DB -> Migration
   Tenant B DB -> Migration
   Tenant C DB -> Migration
   ...
   Tenant N DB -> Migration
*/

In [ ]:
/*
--------------------------------------------------------
3.2 SEPARATE SCHEMAS (Ek DB, Alag Schema Per Tenant)
--------------------------------------------------------

Is model mein ek hi database instance hota hai,
lekin har tenant ke liye alag schema hota hai.

Matlab:

- Tenant A -> tenant_acme schema
- Tenant B -> tenant_globex schema
- Tenant C -> tenant_initech schema

Har schema ke andar same type ki tables ho sakti hain,
jaise users, orders, products.


TEXT DIAGRAM:

              [ One Database Instance ]
                       |
          --------------------------------
          |              |               |
          v              v               v
   tenant_acme      tenant_globex    tenant_initech
       |                 |                |
   users              users            users
   orders             orders           orders
   products           products         products


TENANT KA SCHEMA SELECT KARNA:
--------------------------------------------------------

PostgreSQL mein "search_path" set karke decide karte hain
ki current request kis tenant ke schema ke saath kaam karegi.

Example:

SET search_path TO tenant_acme;

Ab database queries tenant_acme schema ke tables ko use karengi.


REQUEST FLOW:

Request
   |
   v
X-Tenant-ID: acme
   |
   v
[Routing Logic]
   |
   v
Tenant ka schema identify karo
   |
   v
SET search_path TO tenant_acme
   |
   v
[tenant_acme schema]
   |
   +-- users
   +-- orders
   +-- products


IMPORTANT: CONNECTION POOLING
--------------------------------------------------------

Agar connection pooling use kar rahe ho,
to search_path ko request ke baad reset karna important hai.

Kyun?

Connection pool mein database connection reuse hota hai.

Example:

Request 1:
Tenant A
   |
   v
search_path = tenant_acme

Connection pool mein connection wapas gaya


Request 2:
Tenant B
   |
   v
Agar search_path reset nahi hua
   |
   v
Galti se tenant_acme ka context use ho sakta hai

Isliye tenant context ko correctly reset/replace karna zaroori hai.

Goal:

Tenant A -> Sirf tenant_acme
Tenant B -> Sirf tenant_globex


IMPORTANT: SCHEMA NAME TRUSTED HONA CHAHIYE
--------------------------------------------------------

Schema ka naam directly URL ya user input se nahi lena chahiye.

Wrong approach:

URL/User Input
      |
      v
schema name directly use karo

Isse security problem ho sakti hai.

Correct approach:

Request
   |
   v
Tenant ID
   |
   v
Trusted Metadata / Tenant Mapping
   |
   v
Correct Schema Name
   |
   v
Database


ACHA (PROS):
--------------------------------------------------------

1. Good Database-Level Isolation
   -> Har tenant ka separate schema hota hai
   -> Database level par achi isolation milti hai

2. Ek Hi Database Instance Manage Karna
   -> Multiple database instances ke bajay
      ek database instance manage karna hota hai

3. Cross-Tenant Analytics Easier
   -> Sab schemas same database instance mein hain
   -> Isliye cross-tenant analytics separate databases
      ke comparison mein relatively easier ho sakta hai


BURA (CONS):
--------------------------------------------------------

1. N Schemas Manage Karne Padte Hain
   -> Har tenant ka separate schema hota hai
   -> N tenants = N schemas

2. Schema Count Limit
   -> Kuch databases mein bahut large number of schemas
      manage karna practical ya supported nahi ho sakta

3. Migrations Per-Tenant
   -> Schema change hone par migration ko
      har tenant ke schema par run karna pad sakta hai

   Example:

   tenant_acme    -> Migration
   tenant_globex  -> Migration
   tenant_initech -> Migration
   ...
*/

In [ ]:
/*
--------------------------------------------------------
3.3 SHARED TABLES with tenant_id COLUMN
--------------------------------------------------------

Is model mein sab tenants ek hi database aur
same tables share karte hain.

Har row ke andar ek "tenant_id" column hota hai.

Ye tenant_id batata hai ki particular row
kis tenant ki hai.


TEXT DIAGRAM - users table:

  tenant_id | user_id | name
  ----------|---------|------
  acme      |    1    | Alice
  acme      |    2    | Bob
  globex    |    1    | Carol
  initech   |    1    | Dave


Example:

Alice -> Tenant Acme ki user hai
Bob   -> Tenant Acme ki user hai
Carol -> Tenant Globex ki user hai
Dave  -> Tenant Initech ki user hai


MAIN RULE:
--------------------------------------------------------

Har tenant-scoped database query mein tenant_id
hona ZAROORI hai.

Example:

Tenant Acme ke users chahiye:

SELECT * FROM users
WHERE tenant_id = 'acme';


Agar tenant_id condition nahi lagayi:

SELECT * FROM users;

To sabhi tenants ka data return ho sakta hai.

Isse DATA LEAK ho sakta hai.


ACHA (PROS):
--------------------------------------------------------

1. Sabse Sasta
   -> Sab tenants same tables aur resources share karte hain
   -> Separate database ya schema ki zarurat nahi

2. Ek Hi Schema Manage Karna
   -> Sirf ek database schema maintain karna hota hai

3. Cross-Tenant Analytics Easy
   -> Sab tenants ka data same tables mein hai
   -> Isliye combined analytics relatively easy hota hai

4. Ek Hi Migration
   -> Table/schema change ke liye
      ek hi migration chalani hoti hai


BURA (CONS):
--------------------------------------------------------

1. Sabse Weak Isolation
   -> Sab tenants same tables aur rows share karte hain
   -> Isliye isolation comparatively weak hoti hai

2. Har Jagah tenant_id Enforce Karna Padta Hai
   -> Har tenant-specific query mein tenant_id
      correctly use karna zaroori hai

3. Bug Se Data Leak Ho Sakta Hai
   -> Agar developer query mein tenant_id bhool gaya,
      to ek tenant doosre tenant ka data dekh sakta hai

4. Large Tables Complex Indexing Maangti Hain
   -> Bahut saare tenants ka data same table mein hota hai
   -> Table bahut large ho sakti hai
   -> Performance maintain karne ke liye proper indexing
      important ho jati hai


IMPORTANT SECURITY POINT:
--------------------------------------------------------

Yahi wo model hai jahan ek "normal coding bug"
security incident ban sakta hai.

Example:

Correct:

SELECT * FROM orders
WHERE tenant_id = 'acme';


Bug:

SELECT * FROM orders;

Result:
-> Acme ke orders
-> Globex ke orders
-> Initech ke orders
-> Sab return ho sakte hain


ISOLATION KE LIYE 3 LAYERS:
--------------------------------------------------------

1. Application-Level Checks
   -> Application ensure kare ki request ka tenant
      sirf apna data access kare

              +
              
2. Database Row-Level Security (RLS)
   -> Database level par bhi tenant ke data ko
      restrict karo

              +

3. Cross-Tenant Tests
   -> Tests mein verify karo ki Tenant A
      kabhi Tenant B ka data access na kar sake


Final Architecture:

        [Request]
            |
            v
    [Tenant Context]
            |
            v
 [Application-Level Check]
            |
            v
      [Database Query]
            |
            v
   [Row-Level Security]
            |
            v
      [Tenant's Rows]


Goal:

Tenant A -> Sirf Tenant A ka data
Tenant B -> Sirf Tenant B ka data
Tenant C -> Sirf Tenant C ka data
*/

In [ ]:
/*
--------------------------------------------------------
3.4 COMPARISON - Isolation vs Cost Spectrum
--------------------------------------------------------

Tenant data ko store karne ke 3 main approaches hain:

1. Separate Database
2. Separate Schema
3. Shared Tables + tenant_id


ISOLATION vs EFFICIENCY
--------------------------------------------------------

  MORE ISOLATION                         MORE EFFICIENCY
        |                                      |
        v                                      v

  Separate DB  --->  Separate Schema  --->  Shared Tables + ID

  HIGHER COST                            LOWER COST
        |                                      |
        v                                      v

Simple idea:

Separate DB
-> Isolation sabse strong
-> Cost sabse high

Shared Tables
-> Isolation sabse low
-> Cost sabse low


COMPARISON TABLE
--------------------------------------------------------

Strategy          Isolation   Cost      Complexity   Best For
----------------------------------------------------------------
Separate DB       Highest     Highest   Medium       Enterprise /
                                                     Regulated

Separate Schema   High        Medium    Medium       Mid-market SaaS

Shared Tables     Lowest      Lowest    Low          Self-service /
                                                     High volume


1. SEPARATE DATABASE
--------------------------------------------------------

Isolation:
-> Highest
-> Har tenant ka database alag

Cost:
-> Highest
-> Har tenant ke liye separate DB/resources

Complexity:
-> Medium
-> Multiple databases manage karne padte hain

Best For:
-> Enterprise customers
-> Regulated customers


2. SEPARATE SCHEMA
--------------------------------------------------------

Isolation:
-> High
-> Har tenant ka schema alag

Cost:
-> Medium
-> Database instance shared hai

Complexity:
-> Medium
-> Multiple schemas manage karne padte hain

Best For:
-> Mid-market SaaS


3. SHARED TABLES + tenant_id
--------------------------------------------------------

Isolation:
-> Lowest
-> Sab tenants same tables share karte hain
-> tenant_id se data separate hota hai

Cost:
-> Lowest
-> Resources maximum share hote hain

Complexity:
-> Low
-> Ek hi schema/tables manage karne hote hain

Best For:
-> Self-service tenants
-> High-volume SaaS


COMMON INTERVIEW ANSWER
--------------------------------------------------------

Self-service tenants
        |
        v
   Pooled / Shared
        |
        v
Shared Tables ya shared infrastructure


Enterprise Tenants
        |
        v
Strong Isolation
        |
        v
Separate DB / Dedicated Pool


IMPORTANT: FLEXIBLE CONTROL PLANE
--------------------------------------------------------

Control Plane ko flexible rakho taaki tenant ko
future mein ek storage model se doosre model mein
move kiya ja sake.

Example:

Initially:

Small Tenant
    |
    v
Shared / Pooled Infrastructure


Later, tenant bada ho gaya:

Small Tenant
    |
    v
Enterprise Tenant
    |
    v
Separate DB / Dedicated Pool


Iska benefit:

Tenant ko initially cheap pooled infrastructure
mil sakta hai, aur growth ke saath usko stronger
isolation wale setup mein move kiya ja sakta hai.


INTERVIEW SUMMARY:
--------------------------------------------------------

"Self-service tenants ko pooled infrastructure mein rakha
ja sakta hai, jabki enterprise tenants ko separate DB ya
dedicated pool diya ja sakta hai. Control plane flexible
rakhenge taaki future mein tenant ko ek pool se doosre
pool ya dedicated infrastructure mein move kiya ja sake."
*/

In [ ]:
/*
--------------------------------------------------------
4. NOISY NEIGHBOR PROBLEM
--------------------------------------------------------

Jab multiple tenants same resources share karte hain,
to ek tenant ka heavy workload doosre tenants ko slow
kar sakta hai.

Is problem ko "Noisy Neighbor" problem kehte hain.


TEXT DIAGRAM:

  Tenant A (Normal Load) --\
                            \
  Tenant B (HEAVY QUERY) ----> [Shared Database]
                            /       |
  Tenant C (Normal Load) --/       |
                                  CPU 100%
                                    |
                    -------------------------------
                    |                             |
              Slow Response                 Slow Response
                 Tenant A                       Tenant C
                    |                             |
                    -------- B ki wajah se --------
                         dono suffer karenge


Simple Example:

Tenant A -> Normal traffic
Tenant B -> Bahut heavy query chala raha hai
Tenant C -> Normal traffic

Tenant B database ke CPU/resources ko bahut zyada
use kar leta hai.

Result:

Tenant A -> Slow response
Tenant B -> Heavy workload
Tenant C -> Slow response

Yaani Tenant B ki wajah se Tenant A aur C bhi affect
ho rahe hain.


COMMON SCENARIOS:
--------------------------------------------------------

1. Heavy Analytics Query
   -> Ek tenant bahut heavy analytics query chalata hai
   -> Database par heavy load ya locks aa sakte hain
   -> Doosre tenants ki queries slow ho sakti hain


2. Bulk Import
   -> Ek tenant bahut large amount mein data import karta hai
   -> Disk I/O ka bahut zyada use ho sakta hai
   -> Doosre tenants ke requests slow ho sakte hain


3. Viral Tenant
   -> Ek tenant suddenly viral ho jata hai
   -> Uske paas bahut zyada traffic aa jata hai
   -> Shared infrastructure par traffic flood ho sakta hai
   -> Doosre tenants bhi affect ho sakte hain


4. Scheduled Jobs Same Time
   -> Sab tenants ke scheduled jobs ek hi time par run ho jaate hain
   -> Suddenly shared resources par load bahut badh jata hai
   -> Multiple tenants slow ho sakte hain


SOLUTIONS:
--------------------------------------------------------

Noisy Neighbor ko solve karne ke liye
multiple layers use karni chahiye.

Matlab sirf ek solution par depend nahi karna chahiye.

Aage hum application, database, infrastructure
aur tenant-level isolation ke through is problem
ko handle kar sakte hain.
*/

In [ ]:
/*
--------------------------------------------------------
4.1 RESOURCE QUOTAS AND RATE LIMITING
--------------------------------------------------------

Noisy Neighbor problem ko control karne ke liye
har tenant ke resources par limits set kar sakte hain.

Har tenant ki tier ke according alag limits ho sakti hain.

Example:

Acme Corp - Standard Tier

  API Requests      : 1000 / minute
  DB Queries        : 500 / minute
  Storage           : 10 GB
  Concurrent Users  : 100
  Export Rows       : 10,000 / request


Matlab:

- Acme ek minute mein maximum 1000 API requests kar sakta hai
- DB par maximum 500 queries/minute allowed hain
- Maximum 10 GB storage use kar sakta hai
- Maximum 100 users ek time par active ho sakte hain
- Ek export request mein maximum 10,000 rows allowed hain


TEXT FLOW:
--------------------------------------------------------

              [Request]
                  |
                  v
        [Extract tenant_id]
                  |
                  v
       [Rate Limit Check]
                  |
          ----------------
          |              |
         YES             NO
          |              |
          v              v
   429 Too Many       [Process
     Requests           Request]


Agar tenant ki limit exceed ho gayi:

Request
   |
   v
Rate Limit Check
   |
   v
Limit Exceeded = YES
   |
   v
429 Too Many Requests


Agar limit exceed nahi hui:

Request
   |
   v
Rate Limit Check
   |
   v
Limit Exceeded = NO
   |
   v
Process Request


RATE LIMIT COUNTERS:
--------------------------------------------------------

Rate limit counters ko usually fast shared store mein
rakha jata hai, jaise Redis.

Example:

Redis

tenant:acme:api_requests
        |
        v
      Counter

Counter ke saath expiry bhi set ki ja sakti hai.

Example:

Acme API Limit = 1000/minute

Counter:
tenant:acme:api_requests
        |
        v
       1000
        |
        v
      Expiry
     60 seconds


Simple idea:

Tenant
   |
   v
Request
   |
   v
tenant_id identify karo
   |
   v
Redis mein tenant ka counter check karo
   |
   v
Limit exceed?
   |
   +---- YES -> 429 Too Many Requests
   |
   +---- NO  -> Counter increase -> Request process


GOAL:
--------------------------------------------------------

Har tenant ko limited resources do taaki:

Tenant A -> Apni limit ke andar resources use kare
Tenant B -> Apni limit ke andar resources use kare
Tenant C -> Apni limit ke andar resources use kare

Isse ek tenant excessive resources consume karke
doosre tenants ko easily affect nahi kar pata.
*/

In [ ]:
/*
--------------------------------------------------------
4.2 QUERY GOVERNANCE
--------------------------------------------------------

Rate limiting sirf ye control karta hai ki tenant
kitni requests bhej sakta hai.

Lekin problem ye hai ki ek single request bhi
bahut heavy ho sakti hai.

Example:

Tenant ne sirf 1 request bheji,
lekin us request mein bahut heavy database query hai.

Result:
-> Database par high load aa sakta hai
-> DB resources consume ho sakte hain
-> Doosre tenants slow ho sakte hain

Isliye sirf "kitni requests" control karna enough nahi hai.

Query ka SHAPE aur COST bhi control karna chahiye.


QUERY GOVERNANCE RULES:
--------------------------------------------------------

1. Max Execution Time
   -> Maximum: 30 seconds

   Agar query 30 seconds se zyada chal rahi hai,
   to usko stop/reject kiya ja sakta hai.

   Goal:
   -> Long-running queries aur long locks ko prevent karna


2. Max Rows Scanned
   -> Maximum: 1,000,000 rows

   Query ko unlimited rows scan karne ki permission
   nahi deni chahiye.

   Goal:
   -> Full table scan ya extremely expensive scan
      ko control karna


3. Max Result Set Size
   -> Maximum: 10,000 rows

   Ek query se bahut bada result return nahi hona chahiye.

   Goal:
   -> Application/database memory ko overload hone se bachana


4. tenant_id WHERE Clause
   -> Required

   Har tenant-scoped query mein tenant_id ka filter
   hona zaroori hai.

   Goal:
   -> Cross-tenant data access prevent karna


5. SELECT * Without LIMIT
   -> Blocked

   Unlimited rows return karne wali query allow nahi karni.

   Example:

   SELECT * FROM orders;

   Is type ki unbounded query ko block karna chahiye.

   Goal:
   -> Bahut bada result set return hone se prevent karna


QUERY GOVERNANCE SUMMARY:
--------------------------------------------------------

Rule                    Limit / Requirement
--------------------------------------------------------
Max execution time      30 seconds
Max rows scanned        1,000,000
Max result set          10,000 rows
tenant_id WHERE         Required
SELECT * without LIMIT  Blocked


WHY IMPORTANT?
--------------------------------------------------------

Ye especially important hai:

- Report Builders
- Data Exports
- Custom Dashboards

Kyunki in features mein users generally
complex aur large queries generate kar sakte hain.

Simple Rule:

Rate Limiting
-> "Kitni requests?"

        +

Query Governance
-> "Request ke andar query kitni expensive hai?"

        |

        v

Better protection against
Noisy Neighbor + Database Overload
*/

In [ ]:
/*
--------------------------------------------------------
4.3 TENANT-AWARE RESOURCE POOLS
--------------------------------------------------------

Noisy Neighbor problem ko reduce karne ke liye
different tenant tiers ke liye alag resource pools
banaye ja sakte hain.

Matlab:

- Standard tenants -> Standard Pool
- Premium/Enterprise tenants -> Premium Pool

Isse dono tiers ke resources directly share nahi karte.


TEXT DIAGRAM:

                    [Load Balancer]
                   /              \
                  /                \
                 v                  v

          [STANDARD POOL]      [PREMIUM POOL]
           |        |           |     |     |
        Server1  Server2     Server1 Server2 Server3
           |        |           |     |     |
           v        v           v     v     v
       [Standard DB]          [Premium DB]
           |                      |
           v                      v
   Standard Tenants          Enterprise Tenants


STANDARD POOL:
--------------------------------------------------------

Standard Tenants
       |
       v
[Standard Pool]
       |
   ----------------
   |              |
[Server1]      [Server2]
   \              /
    \            /
     [Standard DB]


PREMIUM POOL:
--------------------------------------------------------

Enterprise Tenants
       |
       v
[Premium Pool]
       |
   ---------------------------
   |            |            |
[Server1]    [Server2]    [Server3]
   \            |            /
    \           |           /
        [Premium DB]


MAIN IDEA:
--------------------------------------------------------

Agar Standard Tenant ka load bahut high ho jaye:

Standard Tenant
       |
       v
[Standard Pool]
       |
       v
Heavy Load

To iska direct impact Premium Pool ke
resources par nahi padega.

Similarly:

Premium Tenant
       |
       v
[Premium Pool]
       |
       v
Heavy Load

Standard Pool separately operate karega.


BENEFIT:
--------------------------------------------------------

Resource pools ko tenant tier ke according separate
karne se:

Standard Tenants
-> Standard resources share karte hain

Enterprise Tenants
-> Premium/dedicated resources use karte hain

Isse resource isolation improve hoti hai aur
Noisy Neighbor ka impact reduce hota hai.
*/

In [ ]:
/*
--------------------------------------------------------
4.4 FAIR SCHEDULING
--------------------------------------------------------

Rate limiting ke baad bhi problem ho sakti hai.

Kuch tenants allowed limit ke andar rehkar bhi
legitimately bahut heavy traffic generate kar sakte hain.

Agar unki requests continuously process hoti rahein,
to queue mein doosre tenants ki requests wait kar sakti hain.

Isliye request queue mein FAIR SCHEDULING use karte hain.

Ek common approach hai:

WEIGHTED FAIR QUEUING


MAIN IDEA:
--------------------------------------------------------

Har tenant ko ek "weight" diya jata hai.

Weight batata hai ki us tenant ko queue processing
mein kitna proportionate share milega.

Example:

Tenant A -> Weight 1
Tenant B -> Weight 1
Tenant C -> Weight 2

Tenant C ka weight zyada hai,
isliye usko proportionally zyada processing time milega.


TEXT DIAGRAM:

  Tenant A (Weight 1)
      -> req1, req2

  Tenant B (Weight 1)
      -> req1, req2, req3

  Tenant C (Weight 2)
      -> req1

              |
              v

      [Request Queue]
      [Weighted Fair]
              |
              v

  A1, B1, C1, C2, A2, B2 ...
              |
              v
      [Queue Processor]
              |
              v
           Process


WEIGHT KA MATLAB:
--------------------------------------------------------

Weight 1
-> Normal share of processing

Weight 2
-> Weight 1 wale tenant ke comparison mein
   proportionally zyada processing share


Example:

Tenant A -> Weight 1
Tenant B -> Weight 1
Tenant C -> Weight 2

C ko A/B ke comparison mein approximately
2x processing share mil sakta hai,
depending on scheduler implementation.


IMPORTANT BENEFIT:
--------------------------------------------------------

Premium customers ko higher weight diya ja sakta hai.

Premium Tenant
      |
      v
Higher Weight
      |
      v
More Processing Share


Free/Standard Tenant
      |
      v
Lower Weight
      |
      v
Smaller Processing Share


Lekin Standard/Free tenants ki requests
completely block nahi hoti.

Wo bhi queue mein progress karte rehte hain.


GOAL:
--------------------------------------------------------

Heavy traffic wale premium tenants ko
unke tier ke according zyada processing time do,

BUT

Free/Standard tenants ko completely starve
mat hone do.

Matlab:

Premium Tenant
-> More processing share

Standard Tenant
-> Less processing share

But:

Dono ko eventually processing milti rahe.


SIMPLE RULE:
--------------------------------------------------------

Rate Limiting
-> Tenant kitni requests bhej sakta hai?

        +

Fair Scheduling
-> Available processing capacity ko
   tenants ke beech fairly distribute kaise karein?

        |

        v

Noisy Neighbor ka impact reduce karo
aur kisi tenant ko permanently starve
hone se prevent karo.
*/

In [ ]:
/*
--------------------------------------------------------
5. TENANT IDENTITY AND ROUTING
--------------------------------------------------------

Data ko isolate karne se pehle sabse important question hai:

"Ye request kis tenant ki hai?"

Har incoming request ko reliably identify karna
zaroori hai ki wo kis tenant se aa rahi hai.

Example:

Request
   |
   v
"Ye request kis tenant ki hai?"
   |
   v
Tenant A / Tenant B / Tenant C
   |
   v
Us tenant ka data/resources use karo


IMPORTANT:
--------------------------------------------------------

Tenant identity TRUSTED tarike se identify honi chahiye.

Sirf user ke input par blindly trust nahi karna chahiye.

Kyunki agar system galat tenant identify kar de,
to ek tenant doosre tenant ka data access kar sakta hai.


BASIC FLOW:
--------------------------------------------------------

              [Incoming Request]
                      |
                      v
              [Identify Tenant]
                      |
                      v
               [Trusted Tenant ID]
                      |
                      v
              [Tenant Context]
                      |
                      v
       [Database / Resource Routing]
                      |
                      v
             [Tenant's Data]


MAIN IDEA:
--------------------------------------------------------

Tenant Identity
      |
      v
"Kaunsa tenant?"
      |
      v
Tenant Context
      |
      v
"Is request mein kis tenant ka
data/resources use karne hain?"
      |
      v
Routing + Data Isolation


Aage ke sections mein dekhenge ki tenant ko
trusted tarike se identify karne ke different
methods kya hain aur request ko correct
tenant resources tak kaise route kiya jata hai.
*/

In [ ]:
/*
--------------------------------------------------------
4.2 QUERY GOVERNANCE
--------------------------------------------------------

Rate limiting sirf ye control karta hai ki
tenant kitni requests bhej sakta hai.

Lekin ek problem aur hai:

Ek tenant limited number of requests bhejkar bhi
bahut heavy database query chala sakta hai.

Example:

Tenant -> Sirf 1 request
          |
          v
     Very Heavy Query
          |
          v
   Database Overload

Isliye sirf request count control karna enough nahi hai.

Query ka SHAPE aur COST bhi control karna zaroori hai.


QUERY GOVERNANCE RULES:
--------------------------------------------------------

1. Max Execution Time
   -> Maximum: 30 seconds

   Query 30 seconds se zyada nahi chalni chahiye.

   Goal:
   -> Long-running queries ko control karna
   -> Long locks ko prevent karna


2. Max Rows Scanned
   -> Maximum: 1,000,000 rows

   Query ko unlimited rows scan karne ki permission
   nahi deni chahiye.

   Goal:
   -> Expensive/full table scans ko control karna


3. Max Result Set Size
   -> Maximum: 10,000 rows

   Ek query se bahut bada result return nahi hona chahiye.

   Goal:
   -> Application/database memory ko overload hone
      se bachana


4. tenant_id WHERE Clause
   -> Required

   Har tenant-scoped query mein tenant_id ka filter
   hona zaroori hai.

   Goal:
   -> Cross-tenant data access prevent karna


5. SELECT * Without LIMIT
   -> Blocked

   Unlimited rows return karne wali query ko block karo.

   Example:

   SELECT * FROM orders;

   Problem:
   -> Bahut large result set aa sakta hai
   -> Memory aur database resources consume ho sakte hain


QUERY GOVERNANCE SUMMARY:
--------------------------------------------------------

Rule                    Limit / Requirement
--------------------------------------------------------
Max execution time      30 seconds
Max rows scanned        1,000,000
Max result set          10,000 rows
tenant_id WHERE         Required
SELECT * without LIMIT  Blocked


KAHAN ESPECIALLY IMPORTANT HAI?
--------------------------------------------------------

Query Governance especially important hai:

- Report Builders
- Data Exports
- Custom Dashboards

Kyunki in features mein users custom aur
heavy queries generate kar sakte hain.


SIMPLE DIFFERENCE:
--------------------------------------------------------

Rate Limiting
-> "Tenant kitni requests bhej sakta hai?"

Query Governance
-> "Ek request ke andar query kitni expensive ho sakti hai?"

        |
        v

Dono ko saath use karne se
Database Overload aur Noisy Neighbor
problem ko better control kiya ja sakta hai.
*/

In [ ]:
/*
--------------------------------------------------------
5.1 IDENTIFICATION METHODS
--------------------------------------------------------

Data isolate karne se pehle system ko ye pata hona chahiye:

"Ye request kis tenant ki hai?"

Tenant ko identify karne ke multiple methods hain.


(a) SUBDOMAIN-BASED
--------------------------------------------------------

URL:

https://acme.slack.com/messages

Yahan:

acme = Tenant ID / Tenant Name


FLOW:

Request
   |
   v
Parse Subdomain
   |
   v
"acme"
   |
   v
Lookup Tenant
   |
   v
Set Tenant Context
   |
   v
Process Request


Example:

acme.example.com -> Acme Tenant
globex.example.com -> Globex Tenant


ACHA:
-> URL clean hoti hai
-> Tenant ka branding better ho sakta hai


BURA:
-> Wildcard DNS setup chahiye
-> SSL certificates manage karne padte hain
-> Subdomain conflicts handle karne padte hain



(b) PATH-BASED
--------------------------------------------------------

URL:

api.example.com/tenants/acme/orders

Yahan:

/tenants/acme/
       |
       v
    Tenant ID


FLOW:

Request
   |
   v
Parse URL Path
   |
   v
"acme"
   |
   v
Lookup Tenant
   |
   v
Set Tenant Context
   |
   v
Process Request


ACHA:
-> Simple approach
-> Ek hi domain use hota hai
-> DNS ki extra complexity nahi


BURA:
-> User ke liye URL less intuitive ho sakti hai
-> URL comparatively lambi hoti hai



(c) HEADER-BASED
--------------------------------------------------------

Request:

GET /api/orders

X-Tenant-ID: acme

Authorization: Bearer ...


Yahan:

X-Tenant-ID
     |
     v
   acme
     |
     v
Tenant ID


FLOW:

Request
   |
   v
Read X-Tenant-ID
   |
   v
Validate:
Tenant ID token ke tenant se match karta hai?
   |
   +---- NO ----> Reject Request
   |
   +---- YES ---> Set Tenant Context
                       |
                       v
                 Process Request


ACHA:
-> URLs clean rehti hain
-> Programmatic/API access ke liye flexible hai


BURA:
-> Custom header ko client ko correctly handle karna padta hai
-> Client header bhejna bhool sakta hai



(d) TOKEN-BASED (JWT CLAIMS)
--------------------------------------------------------

JWT ke andar tenant information store ki ja sakti hai.

JWT Payload:

{
  sub: "user_123",
  tenant_id: "acme",
  roles: "admin"
}


Yahan:

sub
-> User ki identity

tenant_id
-> User kis tenant se belong karta hai

roles
-> User ka role


FLOW:

Request
   |
   v
Validate JWT
   |
   v
Extract tenant_id
   |
   v
Set Tenant Context
   |
   v
Process Request


Example:

JWT
 |
 +-- sub       -> user_123
 |
 +-- tenant_id -> acme
 |
 +-- roles     -> admin


ACHA:
-> Self-contained hota hai
-> Tenant information token ke andar available hoti hai
-> Agar JWT properly signed hai, to claims
   tamper-evident hote hain


BURA:
-> Token ka size increase hota hai
-> Agar user tenant switch kare,
   to token refresh ki zarurat ho sakti hai
-> Token revocation ko properly handle karna padta hai



QUICK COMPARISON:
--------------------------------------------------------

Method              Main Idea
--------------------------------------------------------
Subdomain           acme.example.com
Path                example.com/tenants/acme
Header              X-Tenant-ID: acme
JWT Claims          tenant_id: acme


SIMPLE IDEA:
--------------------------------------------------------

Request
   |
   v
[Identify Tenant]
   |
   +--> Subdomain
   |
   +--> Path
   |
   +--> Header
   |
   +--> JWT
   |
   v
[Set Tenant Context]
   |
   v
[Route to Correct Tenant Data/Resources]
*/

In [ ]:
/*
--------------------------------------------------------
5.2 TENANT CONTEXT PATTERN
--------------------------------------------------------

Tenant identify hone ke baad sirf request ke starting
mein tenant ID pata hona enough nahi hai.

Tenant Context ko poori request journey mein carry
karna chahiye.

Matlab request ke har important layer ko pata hona chahiye
ki request kis tenant ki hai.

Tenant Context in sab layers mein available hona chahiye:

- Authentication
- Business Logic
- Database
- Cache
- Logs
- Rate Limits
- Background Jobs


TEXT FLOW:
--------------------------------------------------------

  [Request]
      |
      v
  [Middleware]
  Set Tenant Context
      |
      v
  [Controller]
  Pass Tenant Context
      |
      v
  [Service]
  Pass Tenant Context
      |
      v
  [Data Layer]
  Use Tenant Context


TENANT CONTEXT:
--------------------------------------------------------

Context = {
    tenant_id: "acme",
    tier: "premium",
    limits: ...
}


Example:

Request
   |
   v
tenant_id = acme
tier = premium
limits = ...
   |
   v
[Auth]
   |
   v
[Business Logic]
   |
   v
[Database]
   |
   v
[Cache]
   |
   v
[Logs / Rate Limits / Jobs]


MAIN IDEA:
--------------------------------------------------------

Tenant Context ko request ke saath consistently
carry karna zaroori hai.

Agar kisi layer mein tenant context missing ya wrong hua,
to wrong tenant ka data/resource access hone ka risk hai.


TENANT CONTEXT IMPLEMENT KARNE KE 3 PATTERNS:
--------------------------------------------------------


1. THREAD-LOCAL STORAGE
--------------------------------------------------------

Tenant context ko ek thread-local variable mein
store kiya jata hai.

Flow:

Request
   |
   v
Tenant ID
   |
   v
Thread-Local Storage
   |
   v
Different functions
   |
   v
Tenant Context access


ACHA:
-> Simple approach

BURA:
-> Async operations ke saath break ho sakta hai


2. EXPLICIT PARAMETER PASSING
--------------------------------------------------------

tenant_id ko har function mein explicitly
parameter ke form mein pass karte hain.

Example:

function getOrders(tenant_id) {
    ...
}

function processOrder(tenant_id, order) {
    ...
}


Flow:

Controller
   |
   | tenant_id
   v
Service
   |
   | tenant_id
   v
Data Layer


ACHA:
-> Explicit hota hai
-> Async operations ke saath achhe se kaam karta hai

BURA:
-> Verbose hota hai
-> tenant_id ko baar-baar pass karna padta hai


3. REQUEST-SCOPED DEPENDENCY INJECTION
--------------------------------------------------------

Tenant Context ko request-scoped service ki tarah
inject kiya jata hai.

Flow:

Request
   |
   v
Tenant Context
   |
   v
Request-Scoped Service
   |
   +----> Controller
   |
   +----> Service
   |
   +----> Data Layer


ACHA:
-> Clean approach
-> Testable hota hai

BURA:
-> Framework par depend karta hai


QUICK COMPARISON:
--------------------------------------------------------

Pattern                    ACHA                    BURA
----------------------------------------------------------------
Thread-Local Storage       Simple                  Async mein break
                                                   ho sakta hai

Explicit Parameter        Explicit + Async        Verbose
Passing                    friendly

Request-Scoped DI         Clean + Testable        Framework dependent


SIMPLE IDEA:
--------------------------------------------------------

Request
   |
   v
Identify Tenant
   |
   v
Create Tenant Context
   |
   v
Carry Context Through
   |
   +--> Auth
   +--> Controller
   +--> Service
   +--> Database
   +--> Cache
   +--> Logs
   +--> Rate Limits
   +--> Background Jobs

        |
        v

Har layer ko correct tenant context milna chahiye.
*/

In [ ]:
/*
--------------------------------------------------------
5.3 DATABASE QUERY ENFORCEMENT
--------------------------------------------------------

Multi-tenant system mein sabse dangerous bug ye hai:

"Koi database access path tenant scope bhool jaye."

Example:

Tenant A ki request
      |
      v
[Database Query]
      |
      X
tenant_id filter missing
      |
      v
Tenant B ka data bhi access ho sakta hai

Isliye goal ye hona chahiye:

-> Tenant filtering ko likhna HARD banao
-> Tenant isolation ko test karna EASY banao


APPROACH 1: QUERY WRAPPER
--------------------------------------------------------

Saara database access ek common function/wrapper
ke through karvao.

Ye wrapper tenant_id maangega.

Example:

query(tenant_id, sql)

Agar tenant_id nahi diya:
-> Query ban hi nahi sakti


FLOW:

[Application]
      |
      v
[Query Wrapper]
      |
      |-- tenant_id required
      |
      v
[Database]


BENEFIT:

Developer accidentally tenant_id bhool nahi sakta,
kyunki database access karne ke liye wrapper
tenant_id require karta hai.


APPROACH 2: ROW-LEVEL SECURITY (PostgreSQL)
--------------------------------------------------------

PostgreSQL ka Row-Level Security (RLS) database level
par tenant isolation enforce kar sakta hai.

Matlab:

[Application]
      |
      v
[Database]
      |
      v
[RLS Policy]
      |
      v
Sirf allowed tenant ki rows


Agar application code tenant_id filter lagana bhool bhi jaye,
to database ki RLS policy ensure kar sakti hai ki
wrong tenant ka data return na ho.


IMPORTANT: POOLED CONNECTIONS
--------------------------------------------------------

Agar connection pooling use kar rahe ho,
to tenant context ko carefully handle karna zaroori hai.

Transaction ke andar "SET LOCAL" use kar sakte ho.

SET LOCAL ka benefit:

-> Setting sirf current transaction ke scope mein rahegi
-> Transaction khatam hone ke baad setting automatically
   reset ho jayegi


Alternative:

Agar session-level variable use kar rahe ho,
to ensure karo ki connection reuse hone se pehle
tenant context reliably reset ho.


APPROACH 3: ORM SCOPES
--------------------------------------------------------

ORM mein ek common/base scope bana sakte ho
jisse tenant-scoped queries automatically
current tenant ke according filter hon.

Example idea:

Always filter by current tenant

    getUsers()
        |
        v
    WHERE tenant_id = currentTenant


FLOW:

[Application]
      |
      v
[ORM Base Scope]
      |
      v
tenant_id filter
      |
      v
[Database]


BENEFIT:

-> Har query mein manually tenant_id add karne ki
   zarurat kam ho jati hai
-> Tenant filtering ko common layer mein enforce
   kiya ja sakta hai


RECOMMENDED APPROACH:
--------------------------------------------------------

Sirf ek layer par depend mat karo.

Teeno layers ko combine karo:

        [Application]
             |
             v
   [Tenant-Aware Data Layer]
             |
             v
        [Database]
             |
             v
     [Row-Level Security]
             |
             v
   [Cross-Tenant Tests]


1. Tenant-Aware Data Access Layer
   -> Application level par tenant_id enforce karo

2. Database RLS
   -> Database level par bhi tenant isolation enforce karo

3. Cross-Tenant Tests
   -> Test karo ki Tenant A kabhi Tenant B ka
      data access na kar sake


FINAL RULE:
--------------------------------------------------------

Tenant-aware data access layer
            +
Database Row-Level Security
            +
Cross-tenant tests
            |
            v
     Stronger Tenant Isolation
*/

In [ ]:
/*
--------------------------------------------------------
6. SECURITY CONSIDERATIONS
--------------------------------------------------------

Multi-tenant system mein security ka main goal hai:

"Tenant boundary ko explicit aur testable banana."

Matlab system ko clearly pata hona chahiye ki
kaunsi request kis tenant ki hai.


MAIN SECURITY IDEA:
--------------------------------------------------------

Har layer ko Tenant Context pata hona chahiye.

Example:

Request
   |
   v
[Identify Tenant]
   |
   v
[Tenant Context]
   |
   +----> Authentication
   |
   +----> Business Logic
   |
   +----> Database
   |
   +----> Cache
   |
   +----> Logs
   |
   +----> Rate Limits
   |
   +----> Background Jobs


TENANT BOUNDARY:
--------------------------------------------------------

Tenant A
   |
   v
Sirf Tenant A ka data/resources
   |
   X
Tenant B ka data/resources


Agar koi request tenant boundary cross karne ki
koshish kare:

Request
   |
   v
[Security Check]
   |
   v
Boundary Cross?
   |
   +---- YES ----> REJECT REQUEST
   |
   +---- NO -----> CONTINUE


GOAL:
--------------------------------------------------------

1. Har request ka correct tenant identify ho
2. Har layer mein correct tenant context available ho
3. Tenant sirf apne allowed data/resources access kare
4. Boundary cross karne wali request reject ho
5. Ye behavior tests ke through verify kiya ja sake


SIMPLE RULE:
--------------------------------------------------------

Correct Tenant Identity
        +
Tenant Context
        +
Access Control
        +
Database Isolation
        +
Cross-Tenant Tests
        |
        v
   Secure Tenant Boundary
*/

In [ ]:
/*
--------------------------------------------------------
6.1 DATA LEAKAGE PREVENTION
--------------------------------------------------------

Multi-tenant system mein sabse risky security bug hai:

"Tenant A ka data Tenant B ko dikh jana."

Example:

Tenant A
   |
   v
Request
   |
   v
[Database]
   |
   X
Tenant B ka data mil gaya

Is problem ko sirf ek "query convention" mat samjho.

Isko proper AUTHORIZATION BOUNDARY treat karo.

Matlab:

Tenant A
   |
   v
Sirf Tenant A ka data
   |
   X
Tenant B ka data


DATA LEAKAGE PREVENT KARNE KE LIYE MULTIPLE LAYERS:
--------------------------------------------------------


1. TENANT-AWARE DATA ACCESS
--------------------------------------------------------

Base Repository / ORM Scope ko tenant_id maangna chahiye.

Flow:

[Application]
      |
      v
[Base Repository / ORM]
      |
      v
tenant_id required
      |
      v
[Database]

Goal:
-> Developer accidentally tenant scope na bhool sake


2. ROW-LEVEL SECURITY (RLS)
--------------------------------------------------------

Database level par bhi tenant isolation enforce karo.

Flow:

[Application Query]
       |
       v
   [Database]
       |
       v
   [RLS Policy]
       |
       v
Sirf allowed tenant ki rows
       |
       v
    Response

Agar application tenant filter bhool bhi jaye,
database RLS additional protection provide kar sakta hai.


3. CODE REVIEW CHECKLIST
--------------------------------------------------------

Code review ke time check karo:

-> Database queries mein tenant scoping hai?
-> Cache keys tenant-aware hain?
-> Background jobs mein tenant context hai?
-> Kisi bhi access path se cross-tenant data
   accidentally accessible to nahi hai?


4. AUTOMATED TESTING
--------------------------------------------------------

Tests se verify karo ki ek tenant
doosre tenant ka data access nahi kar sakta.

Example:

Tenant A
   |
   v
Request for Tenant B data
   |
   v
[Access Check]
   |
   v
ACCESS DENIED


Important test:

Tenant A -> Tenant A data = ALLOWED
Tenant A -> Tenant B data = DENIED

Tenant B -> Tenant B data = ALLOWED
Tenant B -> Tenant A data = DENIED


5. SECURITY TESTING
--------------------------------------------------------

Regular security audits mein bhi
tenant isolation ko test karna chahiye.

Check karo:

-> Kya Tenant A Tenant B ka data access kar sakta hai?
-> Kya koi API endpoint tenant boundary bypass kar sakta hai?
-> Kya cache ya background jobs se data leak ho sakta hai?


OVERALL SECURITY LAYERS:
--------------------------------------------------------

        [Tenant-Aware Data Access]
                    |
                    v
             [Row-Level Security]
                    |
                    v
              [Code Review]
                    |
                    v
           [Automated Testing]
                    |
                    v
            [Security Testing]
                    |
                    v
       Strong Tenant Isolation


FINAL RULE:
--------------------------------------------------------

Tenant isolation ko ek single check par depend
mat karo.

Application Layer
       +
Database Layer
       +
Code Review
       +
Automated Tests
       +
Security Testing
       |
       v
Data Leakage Risk ko reduce karo.
*/

In [ ]:
/*
--------------------------------------------------------
6.2 AUTHENTICATION AND AUTHORIZATION
--------------------------------------------------------

Authentication aur Authorization do different concepts hain.


AUTHENTICATION:
--------------------------------------------------------

Authentication ka matlab:

"Tum kaun ho?"

Yaani system verify karta hai ki request bhejne wala
user actually kaun hai.

Example:

User
  |
  v
Login
  |
  v
[Authentication]
  |
  v
User Identity Verified


AUTHORIZATION:
--------------------------------------------------------

Authorization ka matlab:

"Tum is tenant ke andar ye action kar sakte ho kya?"

Yaani user ki identity verify hone ke baad check hota hai
ki user ko particular tenant ke andar particular action
karne ki permission hai ya nahi.

Example:

User = user_123
Tenant = Acme
Role = Admin

Ab system check karega:

-> Kya user_123 Acme tenant ka member hai?
-> Kya user_123 ko ye action karne ki permission hai?


TEXT FLOW:
--------------------------------------------------------

              [Request]
                  |
                  v
          [Authentication]
                  |
          -----------------
          |               |
       Invalid           Valid
          |               |
          v               v
       [401]         [Tenant Match?]
   Unauthorized            |
                     ---------------
                     |             |
                    No            Yes
                     |             |
                     v             v
                  [403]      [Authorization]
               Forbidden           |
                              ---------------
                              |             |
                           Denied         Allowed
                              |             |
                              v             v
                           [403]      [Process Request]
                        Forbidden


3 STEPS ZAROORI:
--------------------------------------------------------

STEP 1: AUTHENTICATE THE USER

Question:
"Tum kaun ho?"

-> User ki identity verify karo
-> Invalid identity -> 401 Unauthorized


STEP 2: VERIFY TENANT MEMBERSHIP

Question:
"Tum is tenant ke member ho?"

Example:

Authenticated User
       |
       v
Tenant = Acme
       |
       v
Check membership
       |
       +---- Not a member -> 403 Forbidden
       |
       +---- Member ------> Continue


STEP 3: AUTHORIZE THE ACTION

Question:
"Tumhe ye action karne ki permission hai?"

Example:

User
  |
  v
Acme Tenant
  |
  v
Role / Permission Check
  |
  +---- Permission nahi -> 403 Forbidden
  |
  +---- Permission hai -> Process Request


COMMON MISTAKE:
--------------------------------------------------------

Sirf URL ya Header se tenant identify kar lena
without checking ki authenticated user actually
us tenant ka member hai ya nahi.


WRONG FLOW:

URL:
 /tenants/acme/orders

       |
       v

Tenant = "acme"
       |
       v
Directly data access
       X
Membership check missing


CORRECT FLOW:

Request
   |
   v
Authenticate User
   |
   v
Identify Tenant
   |
   v
Verify User belongs to Tenant
   |
   v
Check Permission
   |
   v
Process Request


SIMPLE RULE:
--------------------------------------------------------

Authentication
-> "Tum kaun ho?"

        +

Tenant Membership
-> "Tum kis tenant ke member ho?"

        +

Authorization
-> "Tum kya kar sakte ho?"

        |
        v

Secure Tenant Access
*/

In [ ]:
/*
--------------------------------------------------------
6.3 ENCRYPTION
--------------------------------------------------------

Multi-tenant system mein sensitive data ko protect karne
ke liye encryption important hai.

Encryption ko 3 areas mein samjho:

1. At Rest
2. In Transit
3. In Use


1. AT REST
--------------------------------------------------------

At Rest ka matlab:

Data jab database, storage ya disk mein stored hai.

Is data ko encrypted rakhna chahiye.

Recommended approach:

-> Envelope Encryption
-> Strict isolation requirements wale tiers ke liye
   tenant-specific encryption keys use ki ja sakti hain


2. IN TRANSIT
--------------------------------------------------------

In Transit ka matlab:

Data jab network ke through travel kar raha hai.

Example:

Client
   |
   | Encrypted Connection
   v
Server
   |
   | Encrypted Connection
   v
Database


Rule:

-> TLS everywhere


3. IN USE
--------------------------------------------------------

In Use ka matlab:

Data jab application/process ke andar use ho raha hai.

Is stage par:

-> Sensitive data ka exposure minimize karo
-> Sensitive data ko logs mein kabhi mat likho


Example:

WRONG:

logger.info({
    password: user.password,
    creditCard: user.creditCard
});


Better:

logger.info({
    userId: user.id
});


TENANT KEY DECRYPTION FLOW:
--------------------------------------------------------

                [Master Key in HSM]
                         |
                      decrypts
                         |
                         v
              [Encrypted Tenant Keys]
                  (stored encrypted)
                         |
                      decrypts
                         |
                         v
                  [Tenant Data]


Step-by-step:

1. Master Key
   -> HSM mein securely stored hoti hai

2. Encrypted Tenant Keys
   -> Tenant-specific keys encrypted form mein stored hoti hain

3. Master Key
   -> Encrypted tenant key ko decrypt karne mein help karti hai

4. Tenant Key
   -> Tenant ke encrypted data ko decrypt karne ke liye use hoti hai

5. Tenant Data
   -> Decrypted data application use kar sakti hai


WHY THIS PATTERN?
--------------------------------------------------------

Envelope encryption ka benefit:

-> Key exposure ka blast radius limit kiya ja sakta hai
-> Har tenant ke liye key rotation support ki ja sakti hai

Example:

Tenant A -> Key A
Tenant B -> Key B
Tenant C -> Key C


Agar Tenant A ki key rotate karni hai:

Tenant A
   |
   v
Old Key A
   |
   v
New Key A

Baaki tenants ki keys unaffected reh sakti hain.


TRADE-OFF:
--------------------------------------------------------

Benefit:
-> Better isolation
-> Blast radius limit
-> Per-tenant key rotation support


Cost:
-> Key management complexity increase hoti hai


SIMPLE RULE:
--------------------------------------------------------

At Rest
-> Encrypt stored data
-> Envelope encryption
-> Strict tiers ke liye tenant-specific keys

In Transit
-> TLS everywhere

In Use
-> Sensitive data exposure minimize karo
-> Sensitive data logs mein mat rakho
*/

In [ ]:
/*
--------------------------------------------------------
6.4 AUDIT LOGGING
--------------------------------------------------------

Multi-tenant system mein audit logs ke andar bhi
Tenant Context hona zaroori hai.

Kyun?

Taaki pata chal sake:

-> Kis tenant ke andar action hua?
-> Kis tenant se related incident hai?
-> Support case kis tenant ka hai?
-> Compliance review kis tenant ke scope mein hai?


AUDIT LOG EXAMPLE:
--------------------------------------------------------

  tenant_id | user_id   | action        | resource
  ----------|-----------|---------------|----------
  acme      | user_123  | ORDER_CREATED | order_101
  globex    | user_456  | ORDER_DELETED | order_202


Yahan har audit event ke saath tenant_id stored hai.

Isse audit logs ko tenant ke according
properly filter/scope kiya ja sakta hai.


TEXT FLOW:
--------------------------------------------------------

  [User Action]
        |
        v
  [Tenant Context]
        |
        v
  [Audit Logger]
        |
        v
  [Audit Log]
        |
        +--> tenant_id
        +--> user_id
        +--> action
        +--> resource


WHY TENANT CONTEXT IMPORTANT?
--------------------------------------------------------

1. INCIDENTS

Agar security incident hua:

Incident
   |
   v
tenant_id = acme
   |
   v
Sirf Acme ke relevant audit events identify karo


2. SUPPORT CASES

Support issue:

"Acme tenant ke data mein kya hua?"

   |
   v
tenant_id = acme
   |
   v
Relevant audit logs


3. COMPLIANCE REVIEW

Compliance review ke time:

Tenant
   |
   v
Tenant-specific audit logs
   |
   v
Review


IMPORTANT SECURITY RULE:
--------------------------------------------------------

Audit logs ko bhi normal product data ki tarah
tenant isolation milni chahiye.


Example:

Tenant A
   |
   v
Sirf Tenant A ke audit logs
   |
   X
Tenant B ke audit logs


Tenant B
   |
   v
Sirf Tenant B ke audit logs
   |
   X
Tenant A ke audit logs


FINAL RULE:
--------------------------------------------------------

Audit logs mein tenant context zaroor rakho.

Aur:

Tenant A -> Tenant A ke audit logs
Tenant B -> Tenant B ke audit logs

Ek tenant ko doosre tenant ke audit logs
access nahi karne dene chahiye.
*/

In [ ]:
/*
--------------------------------------------------------
7. SCALING MULTI-TENANT SYSTEMS
--------------------------------------------------------

Multi-tenant system ko scale karne ka main goal hai:

1. App tier ko stateless rakho
2. Tenants ko smartly place karo
3. Tenant ki size/need badhne par usko
   ek infrastructure se doosre infrastructure
   mein move kar sako


1. APP TIER STATELESS RAKHO
--------------------------------------------------------

Application servers ko stateless rakho.

Matlab:

App Server 1
App Server 2
App Server 3

Kisi bhi request ko kisi bhi server par
process kiya ja sake.

TEXT FLOW:

                 [Load Balancer]
                /       |       \
               v        v        v
           [App 1]  [App 2]  [App 3]
              \        |        /
               \       |       /
                [Shared Services]

Agar traffic badhe:

Low Traffic
    |
    v
2 App Servers

        |

High Traffic
    |
    v
5 App Servers


Isse application tier ko horizontally scale karna
easy ho jata hai.


2. TENANTS KO SMARTLY PLACE KARO
--------------------------------------------------------

Sab tenants ko blindly same resources par mat rakho.

Tenant ki:

- Size
- Traffic
- Usage
- Tier
- Resource requirements

ke according unko appropriate pool/database
mein place karo.

Example:

Small Tenants
      |
      v
[Standard Pool]

Large Tenants
      |
      v
[Premium Pool]

Very Large / Regulated Tenant
      |
      v
[Dedicated Infrastructure]


3. TENANT KO MOVE KARNE KI ABILITY
--------------------------------------------------------

Tenant ki requirements time ke saath change ho sakti hain.

Example:

Starting:

Small Tenant
      |
      v
[Shared Pool]


Tenant grow karta hai:

Small Tenant
      |
      v
[Large Tenant]
      |
      v
[Premium Pool / Dedicated DB]


Matlab system mein tenant migration ka path
pehle se design karna chahiye.


OVERALL SCALING FLOW:
--------------------------------------------------------

                    [Load Balancer]
                           |
                           v
                  [Stateless App Tier]
                           |
             ---------------------------
             |            |            |
             v            v            v
        Standard      Premium      Dedicated
          Pool          Pool       Infrastructure
             |            |            |
             v            v            v
        Shared DB     Premium DB   Dedicated DB


SIMPLE RULE:
--------------------------------------------------------

Stateless App
      +
Smart Tenant Placement
      +
Tenant Migration
      |
      v
Scalable Multi-Tenant System
*/

In [ ]:
/*
--------------------------------------------------------
7.1 HORIZONTAL SCALING
--------------------------------------------------------

Horizontal Scaling ka matlab hai:

Ek hi powerful server ko bada karne ke bajay
multiple application servers add karna.

Example:

Low Traffic
    |
    v
[App1]


High Traffic
    |
    v
[App1] [App2] [App3] ... [AppN]


TEXT DIAGRAM:

                 [Load Balancer]
                /      |       \
               v       v        v
            [App1]  [App2]   [AppN]
                \      |       /
                 \     |      /
                    [Cache]
                       |
                       v
                   [Database]


REQUEST FLOW:
--------------------------------------------------------

Client
   |
   v
[Load Balancer]
   |
   +----> App1
   |
   +----> App2
   |
   +----> AppN
              |
              v
           [Cache]
              |
              v
          [Database]


MAIN RULE:
--------------------------------------------------------

Sabhi application servers INTERCHANGEABLE hone chahiye.

Matlab:

Request 1 -> App1
Request 2 -> App2
Request 3 -> App3

Kisi request ko specific server par depend nahi
karna chahiye.


TENANT-SPECIFIC STATE SERVER MEIN MAT RAKHO
--------------------------------------------------------

Kisi particular tenant ka important state
sirf ek application server ke memory mein nahi rakhna chahiye.

Wrong:

Tenant A
   |
   v
[App1 Memory]
   |
   X
App2 ko Tenant A ka state nahi milega


Problem:

Request 1 -> App1
             |
             v
        Tenant State


Request 2 -> App2
             |
             X
        State available nahi


CORRECT APPROACH:
--------------------------------------------------------

Tenant-specific information ko shared/accessible
location mein rakho.

Example:

Request
   |
   v
[App Server]
   |
   +----> Request Context
   |
   +----> Shared Services
   |
   +----> Tenant Metadata Store


Example:

              [Load Balancer]
               /     |     \
              v      v      v
           [App1] [App2] [App3]
              \      |      /
               \     |     /
                [Shared State]
                     |
          -------------------------
          |           |           |
      Tenant       Tenant      Tenant
      Metadata     Context     Config


Tenant Context aur tenant metadata ko
server-specific memory par depend nahi karna chahiye.


SIMPLE RULE:
--------------------------------------------------------

Horizontal Scaling
        |
        v
Multiple App Servers
        |
        v
All Servers Interchangeable
        |
        v
No Tenant-Specific State in One Server
        |
        v
Use Request Context / Shared Services /
Tenant Metadata Store
        |
        v
Easy Horizontal Scaling
*/

In [ ]:
```javascript
/*
--------------------------------------------------------
7.2 DATABASE SCALING
--------------------------------------------------------

Multi-tenant system mein tenants badhne ke saath database par
load bhi badhta hai.

Database ko scale karne ke common approaches:

1. READ REPLICAS
--------------------------------------------------------
                 ┌──────────────┐
                 │   Primary DB │
                 │   (WRITE)    │
                 └──────┬───────┘
                        │ Replication
              ┌─────────┴─────────┐
              ↓                   ↓
       ┌─────────────┐     ┌─────────────┐
       │ Read Replica│     │ Read Replica│
       │   (READ)    │     │   (READ)    │
       └─────────────┘     └─────────────┘

Write requests
      ↓
Primary DB

Read requests
      ↓
Read Replicas


Best for:
- Read-heavy workload
- Jab READ requests bahut zyada ho
- Writes comparatively kam ho

Idea:
Primary DB mainly WRITE handle karta hai,
aur replicas READ load distribute karte hain.


2. TENANT-BASED SHARDING
--------------------------------------------------------

Yahan tenant_id ko shard key banaya jata hai.

                ┌─────────────────┐
                │   Application   │
                └────────┬────────┘
                         │
                    tenant_id
                         ↓
              ┌─────────────────────┐
              │ Shard Router        │
              └─────────┬───────────┘
                        │
          ┌─────────────┼─────────────┐
          ↓             ↓             ↓
      Shard 1        Shard 2       Shard 3
      Tenant A       Tenant C      Tenant E
      Tenant B       Tenant D      Tenant F


Example:

tenant_id = acme
      ↓
Shard 1

tenant_id = globex
      ↓
Shard 2


Yahan tenant_id decide karta hai ki tenant ka data
kis database/shard mein stored hai.


Best for:
- High tenant count
- Bahut saare tenants hone par
- Jab mostly queries ek hi tenant ke data ke liye hoti hain
- Jab cross-tenant queries kam hoti hain


Tenant-based sharding usually achha kaam karta hai
kyunki multi-tenant application mein zyada tar queries
already ek specific tenant ke scope mein hoti hain.

Example:

SELECT * FROM orders
WHERE tenant_id = 'acme';

Is query ko sirf us shard par bheja ja sakta hai
jahan Acme ka data stored hai.

Isse unnecessary shards ko query nahi karna padta.


3. TIERED STORAGE
--------------------------------------------------------

Har data ko same type ke storage par rakhna zaroori nahi hai.

Data ko uski usage/frequency ke according different
storage tiers mein rakha ja sakta hai.


                 DATA
                  │
        ┌─────────┼─────────┐
        ↓         ↓         ↓
       HOT       WARM      COLD
        │         │         │
       SSD      Normal    Archive
      Storage   Storage   Storage


HOT DATA
- Frequently accessed data
- Fast SSD storage par

WARM DATA
- Kabhi-kabhi accessed data
- Normal/cheaper storage par

COLD DATA
- Rarely accessed / old data
- Archive storage mein


Best for:
- Cost optimization
- Large amount of historical data
- Jab old data ko continuously fast storage par
  rakhna expensive ho


--------------------------------------------------------
QUICK SUMMARY
--------------------------------------------------------

Read Replicas
→ Primary DB = WRITE
→ Replicas = READ
→ Best for read-heavy workload


Tenant-based Sharding
→ tenant_id = shard key
→ Different tenants ka data different shards par
→ Best for high tenant count + mostly tenant-scoped queries


Tiered Storage
→ Hot = SSD
→ Warm = normal storage
→ Cold = archive
→ Best for reducing storage cost


IMPORTANT IDEA
--------------------------------------------------------

Multi-tenant systems mein tenant_id already most queries
ka natural scope hota hai.

Isliye tenant-based sharding useful hota hai:

Request
   ↓
tenant_id identify
   ↓
Shard determine
   ↓
Sirf required shard ko query
   ↓
Tenant data


Main benefit:
"Ek tenant ki query ko poore database cluster mein
search karne ki zaroorat nahi padti."
*/
```


In [ ]:
```javascript
/*
--------------------------------------------------------
7.3 TENANT PLACEMENT
--------------------------------------------------------

Multi-tenant system mein bahut saare tenants ko different
shards/servers par place karna padta hai.

Question:
"Kis tenant ko kis shard par rakhein?"

Is decision ko TENANT PLACEMENT kehte hain.


1. ROUND-ROBIN
--------------------------------------------------------

Tenants ko turn-by-turn different shards par distribute karo.

Tenant A → Shard 1
Tenant B → Shard 2
Tenant C → Shard 3
Tenant D → Shard 1
Tenant E → Shard 2
Tenant F → Shard 3

Benefit:
- Even distribution
- Simple implementation

Problem:
- Tenant ka actual size ya activity consider nahi hota.


2. SIZE-BASED PLACEMENT
--------------------------------------------------------

Tenant ke size ke according placement decide hoti hai.

Small Tenant A → Shard 1
Small Tenant B → Shard 1
Large Tenant C → Dedicated Shard 2

Large tenants ko dedicated shard diya ja sakta hai.

Benefit:
- Bade tenants ka load doosre tenants ko affect
  karne se kam hota hai.


3. GEOGRAPHY-BASED PLACEMENT
--------------------------------------------------------

Tenant ko uske users/location ke paas wale region mein
place kiya jata hai.

Example:

India users
    ↓
India Region / Shard

US users
    ↓
US Region / Shard

Europe users
    ↓
Europe Region / Shard

Goal:
- Data users ke paas rahe
- Network distance kam ho
- Latency low rahe


4. ACTIVITY-BASED PLACEMENT
--------------------------------------------------------

Tenant kitna active hai, uske basis par placement decide karo.

Example:

Tenant A → Low activity
Tenant B → High activity
Tenant C → High activity
Tenant D → Low activity

Agar B aur C dono ko same shard par rakh diya,
to shard hotspot ban sakta hai.

Isliye active tenants ko different shards par spread
kiya jata hai.

Goal:
- Active tenants ko distribute karna
- Hotspot avoid karna
- Resource usage balanced rakhna


--------------------------------------------------------
REBALANCING KAB ZAROORI HAI?
--------------------------------------------------------

Tenant placement permanent nahi hoti.

Time ke saath tenants ka size aur activity change ho sakta hai.

Rebalancing tab zaroori ho sakti hai jab:

1. Ek shard hotspot ban jaye
   → Kisi shard par bahut zyada load aa raha ho.

2. Chhota tenant bada ho jaye
   → Tenant ka traffic/data itna grow ho gaya ho
     ki current shard par load badh gaya.

3. Naye shards cluster mein add ho
   → Existing tenants ko naye capacity ke across
     redistribute karna ho.


--------------------------------------------------------
TEXT FLOW - REBALANCING
--------------------------------------------------------

Example:

Tenant A
   ↓
Old Shard
   ↓
Target Shard

Tenant ko old shard se target shard par move karna hai.


STEP 1: CREATE NEW HOME ON TARGET SHARD
--------------------------------------------------------

Target shard par tenant ke liye required
database/table/schema/resources prepare karo.

Old Shard
   └── Tenant A

Target Shard
   └── Tenant A (empty/new home)


STEP 2: SYNC DATA
--------------------------------------------------------

Old shard ka data target shard par copy/sync karo.

Possible approaches:
- CDC (Change Data Capture)
- Dual-write

Old Shard
   ↓
Data Changes
   ↓
Target Shard


Goal:
Target shard ko latest data ke saath synchronize rakhna.


STEP 3: UPDATE ROUTING TABLE
--------------------------------------------------------

Routing system ko batao ki tenant ab target shard par hai.

Before:

Tenant A → Shard 1

After:

Tenant A → Shard 2


Routing Table:

tenant_id    →    shard
------------------------
acme         →    shard-2
globex       →    shard-1


STEP 4: CUTOVER TRAFFIC
--------------------------------------------------------

Ab Tenant A ki new requests ko target shard par bhejna
start karo.

Before:

Tenant A Request
      ↓
   Shard 1

After:

Tenant A Request
      ↓
   Shard 2


Yeh actual traffic switch/cutover hai.


STEP 5: CLEAN UP OLD DATA
--------------------------------------------------------

Target shard par traffic successfully chal raha hai,
data verify ho gaya hai aur system stable hai,

tab old shard se Tenant A ka old data clean up
kiya ja sakta hai.


--------------------------------------------------------
COMPLETE REBALANCING FLOW
--------------------------------------------------------

1. Create new home
       ↓
2. Sync data
   (CDC / Dual-write)
       ↓
3. Update routing table
       ↓
4. Cutover traffic
       ↓
5. Verify + monitor
       ↓
6. Clean up old data


--------------------------------------------------------
GOAL OF REBALANCING
--------------------------------------------------------

Goal hai tenant ko ONLINE ya LOW-DOWNTIME mein move karna.

Simple idea:

COPY
  ↓
VERIFY
  ↓
SWITCH ROUTING
  ↓
MONITOR
  ↓
CLEAN OLD DATA


Important:
Tenant movement ke time application ko unnecessarily
long downtime nahi dena chahiye.

Isliye pehle data copy/sync hota hai,
phir routing switch hoti hai,
phir system monitor karke old data clean kiya jata hai.
*/
```


In [ ]:
```javascript
/*
========================================================
8. INTERVIEW ANSWER FRAMEWORK
========================================================

Interview mein agar "Multi-Tenancy" ka question aaye,
to sirf isolation models ki list mat bolna.

Answer ko pehle RISK BOUNDARIES se start karo.

Interviewer ko dikhana hai ki tum sirf database isolation
nahi, balki complete multi-tenant architecture samajhte ho.


--------------------------------------------------------
STEP 1: TENANT BOUNDARY IDENTIFY KARO
--------------------------------------------------------

Sabse pehle identify karo:

"System ko kaise pata chalta hai ki ye request/job/
cache/log/query kis tenant ka hai?"

Request
   ↓
Tenant Identity
   ↓
Trusted Tenant Context
   ↓
Request / Job / Cache / Query / Log

Important:
Har operation ke saath tenant context clear hona chahiye.


--------------------------------------------------------
STEP 2: SHARING MODEL CHOOSE KARO
--------------------------------------------------------

Tenants ki size aur requirements ke according
sharing model choose karo.

Small Tenants
     ↓
   Pool

Large / Regulated Tenants
     ↓
   Silo / Strong Isolation

Mixed Customer Base
     ↓
   Bridge Model


Simple idea:

Chhote tenants ko pool karo,
bade ya regulated tenants ko stronger isolation do.

Agar dono requirements hain,
to bridge/hybrid model use karo.


--------------------------------------------------------
STEP 3: DATA PEHLE PROTECT KARO
--------------------------------------------------------

Sabse important part:
Tenant data accidentally cross-tenant access nahi hona chahiye.

Isliye explain karo:

- Tenant-scoped queries
- Row-Level Security (RLS)
- Cross-tenant tests
- Code reviews


Flow:

Request
   ↓
Tenant Context
   ↓
Tenant-scoped Query
   ↓
RLS / Database Protection
   ↓
Correct Tenant Data


Goal:
Application ka koi bug accidentally doosre tenant ka
data access na kar sake.


--------------------------------------------------------
STEP 4: NOISY NEIGHBORS CONTROL KARO
--------------------------------------------------------

Shared infrastructure mein ek tenant bahut zyada
resources consume karke doosre tenants ko slow kar sakta hai.

Isko control karne ke liye:

- Rate limits
- Resource quotas
- Query limits
- Fair queues
- Resource pools


Example:

Heavy Tenant
     ↓
Rate Limit / Quota
     ↓
Controlled Resource Usage


Goal:
Ek tenant ka excessive workload doosre tenants ko
unnecessarily affect na kare.


--------------------------------------------------------
STEP 5: TENANT MOVEMENT PLAN KARO
--------------------------------------------------------

Important architectural point:

Tenants hamesha same size ya same traffic par nahi rahenge.

Aaj:

Tenant A → Shard 1

Future mein:

Tenant A → Shard 5


Isliye tenant metadata aur routing information ko
CONTROL PLANE mein rakhna useful hai.

Tenant Metadata
      ↓
Routing Table
      ↓
Current Shard / Tier
      ↓
Tenant Request


Isse tenant ko future mein:

- Shard change
- Tier change
- Resource pool change
- Dedicated infrastructure

par move kiya ja sakta hai.


--------------------------------------------------------
STEP 6: OPERATIONS TENANT-AWARE BANAO
--------------------------------------------------------

Sirf application aur database ko tenant-aware banana
enough nahi hai.

Operations ke important parts mein bhi tenant_id hona chahiye:

- Monitoring
- Audit logs
- Billing
- Support tools
- Migrations


Example:

Request
   ↓
tenant_id
   ├── Database
   ├── Cache
   ├── Queue
   ├── Logs
   ├── Metrics
   ├── Billing
   └── Support


Goal:
Har operational activity ko tenant ke context mein
identify aur manage kar sakein.


========================================================
ACHA INTERVIEW ANSWER KYA COVER KARTA HAI?
========================================================

1. TENANT IDENTITY
--------------------------------------------------------

Tenant identity ko TRUSTED CONTEXT maana jata hai,
raw request parameter nahi.

Wrong approach:

Request → X-Tenant-ID → Direct Database

Better approach:

Request
   ↓
Authenticate User
   ↓
Verify Tenant Membership
   ↓
Trusted Tenant Context
   ↓
Database / Cache / Queue / Logs


--------------------------------------------------------
2. ISOLATION TRADE-OFFS
--------------------------------------------------------

Different isolation models ka trade-off explain karo:

Shared Tables
     ↓
Separate Schema
     ↓
Separate Database
     ↓
Dedicated Infrastructure


Explain:

- Cost
- Isolation
- Complexity
- Scalability
- Compliance
- Customization


--------------------------------------------------------
3. COMPLETE TENANT SCOPE
--------------------------------------------------------

Sirf database queries ki baat mat karo.

Tenant scope ko complete system mein consider karo:

Tenant ID
   ├── Data Access
   ├── Cache Keys
   ├── Queues
   ├── Background Jobs
   ├── Logs
   ├── Support Tools
   └── Migrations


Har jagah tenant context maintain hona chahiye.


--------------------------------------------------------
4. NOISY NEIGHBORS
--------------------------------------------------------

Explain karo ki ek tenant ka excessive resource usage
doosre tenants ko impact kar sakta hai.

Solutions:

- Rate limits
- Quotas
- Query limits
- Fair queues
- Resource pools


--------------------------------------------------------
5. TENANT MOVEMENT
--------------------------------------------------------

Architecture ko tenant movement support karna chahiye.

Tenant size/activity change
          ↓
Need for movement
          ↓
New shard / tier
          ↓
Data migration
          ↓
Routing update
          ↓
Traffic cutover


Important:
Ye assume mat karo ki sab tenants hamesha same size
aur same workload rahenge.


========================================================
INTERVIEW MEIN SIMPLE STRUCTURE
========================================================

Agar interviewer bole:

"Design a multi-tenant system."

To answer ka flow:

1. Tenant Boundary
       ↓
2. Sharing Model
       ↓
3. Data Isolation
       ↓
4. Noisy Neighbor Control
       ↓
5. Tenant Movement
       ↓
6. Tenant-Aware Operations


Ye approach dikhata hai ki tum multi-tenancy ko
sirf "shared database vs separate database" ke form mein
nahi, balki complete system architecture ke form mein
samajhte ho.
*/
```
